# 39 — Does the blend's diversity survive into the blind set, and can it be one object?

**Nothing is trained. No submission is built, validated or sent.** Both questions are answered from
predictions already on disk. `outputs/37_butina_5x5_complete/`,
`outputs/38_cyp3a4_placement_diagnostic/`, `outputs/36_spread_corrected_nb32a/`,
`outputs/34_butina_5x5/`, `outputs/35_butina_repeat3/`, `src/` and their notebooks are read-only and
protected; `notebooks/32_deadzone_aid_retrain.ipynb` is never opened or run, since its candidate-A
submission cell is live uncommented code.

## Part A — the ambiguity decomposition

Krogh & Vedelsby, *Neural Network Ensembles, Cross Validation, and Active Learning* (NIPS 1994),
quoted verbatim from the PDF (Part 0 re-checks these against the file):

> The ambiguity on input x of a single member of the ensemble is defined as
> `a_a(x) = (V_a(x) - V̄(x))²`. The ensemble ambiguity on input x is
> `a(x) = Σ_a w_a a_a(x) = Σ_a w_a (V_a(x) - V̄(x))²`. It is simply the variance of the weighted
> ensemble around the weighted mean, and it measures the disagreement among the networks on
> input x.

> The ambiguity is defined as the variation of the output of ensemble members averaged over
> **unlabeled data**, so it quantifies the disagreement among the networks.

> A combination of the output of several networks (or other predictors) is only useful if they
> disagree on some inputs.

Averaging over the input distribution gives their central identity, `E = Ē - A`, where `E` is the
ensemble's generalisation error, `Ē = Σ_a w_a E_a` the weighted mean of member errors, and
`A = Σ_a w_a A_a` the ensemble ambiguity. Their weights are constrained positive and summing to
one — "The constraint on the sum is crucial for some of the following results" — which notebook
`37`'s greedy weights satisfy by construction, asserted below.

**The reason this is worth doing at all**: `A` needs no labels. So it can be computed on the blind
750 directly, and asks whether the member disagreement that earns the blend its out-of-fold gain is
still present on the compounds that are actually scored.

**CAVEAT, STATED ONCE HERE AND REPEATED AT EVERY TABLE.** The identity is exact for **squared
error** and for nothing else. It does **not** decompose ST-RAE, which is the metric this project is
scored on, nor MAE. Every `A` below is computed on squared error as a **diagnostic of member
disagreement**, never as a decomposition of the scoring metric. ST-RAE figures appear beside it for
context and are never fed through the identity.

## Part B — what a shared member set costs

Notebook `37`'s greedy blend picks a different member set per isoform (3/3/1/3 members; CYP2D6
collapses to `{deadzone: 1.0}` with 25/25 per-fold agreement). A submission built that way is four
ensembles, not one. Part B re-runs the selection under a shared-member-set constraint, keeping
notebook `37`'s nested honesty check so the constrained and unconstrained figures are comparable,
and reports the stricter shared-members-**and**-shared-weights variant as a separate row.

In [1]:
import hashlib
import itertools
import json
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.vendor.openadmet_eval.config import REGRESSION_ENDPOINTS
from src.vendor.openadmet_eval.custom_scoring_functions import rae_soft_threshold_absolute_error

ISOFORMS = [e.split("_")[0] for e in REGRESSION_ENDPOINTS]
PIC50_COL = {iso: f"{iso}_pIC50_direct_inhibition" for iso in ISOFORMS}

OUT = REPO_ROOT / "outputs" / "39_ambiguity_and_shared_blend"
FIG_OUT = OUT / "figures"
for d in [OUT, FIG_OUT]:
    d.mkdir(parents=True, exist_ok=True)

# RUN START recorded at the top, never inferred at the bottom from a file mtime -- the bug
# notebook 38 shipped on its first execution and notebooks 30/34/35/37 already carry the fix for.
RUN_META_PATH = OUT / "run_meta.json"
if RUN_META_PATH.exists():
    RUN_META = json.loads(RUN_META_PATH.read_text())
    print(f"[resume] run started {RUN_META['started_at_utc']}")
else:
    RUN_META = {"started_at": time.time(),
                "started_at_utc": datetime.now(timezone.utc).isoformat()}
    RUN_META_PATH.write_text(json.dumps(RUN_META, indent=2))
    print(f"[new run] started {RUN_META['started_at_utc']}")
RUN_STARTED_AT = float(RUN_META["started_at"])

# READ-ONLY sources.
CURATED = REPO_ROOT / "data" / "processed" / "train_inhibition_curated.csv"
BLIND = REPO_ROOT / "data" / "processed" / "test_blinded_curated.csv"
NB08_OUT = REPO_ROOT / "outputs" / "08_ensemble_selection"
NB10_OUT = REPO_ROOT / "outputs" / "10_final_retrain_predict"
NB11_OUT = REPO_ROOT / "outputs" / "11_caruana_prep"
NB12_OUT = REPO_ROOT / "outputs" / "12_caruana_retrain_predict"
NB30_OUT = REPO_ROOT / "outputs" / "30_aid_full_retrain"
NB37_OUT = REPO_ROOT / "outputs" / "37_butina_5x5_complete"

AUTONOMOUS_DECISIONS = []


def record_decision(decision, reason, alternatives, authority):
    AUTONOMOUS_DECISIONS.append({"decision": decision, "reason": reason,
                                 "alternatives": alternatives, "authority": authority,
                                 "at_utc": datetime.now(timezone.utc).isoformat()})
    (OUT / "decisions_taken.json").write_text(json.dumps(AUTONOMOUS_DECISIONS, indent=2))


def strae(y, p, lo, hi):
    return float(rae_soft_threshold_absolute_error(
        np.asarray(y, float), np.asarray(p, float),
        y_true_upper=np.asarray(hi, float), y_true_lower=np.asarray(lo, float)))


print(f"python {sys.version.split()[0]}  |  OUT: {OUT}")

[new run] started 2026-10-02T11:15:54.013878+00:00
python 3.11.13  |  OUT: /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/39_ambiguity_and_shared_blend


### Part 0 — provenance: the quotations, and notebook 37's blend reproduced

Two checks before anything is read off. The Krogh & Vedelsby quotations are verified verbatim
against the PDF (sha256-pinned), as notebook `37` did for Ash. And notebook `37`'s blend is
**reproduced from its own saved out-of-fold tables** — if the greedy search here does not return
`37`'s published weights and in-sample scores exactly, nothing in Part B is comparable to it.

In [2]:
# ---- (1) the Krogh & Vedelsby quotations, verified against the PDF itself.
KV_SHA256_AT_BUILD = "88de32e9c1d3384970036baf59c7fbcdc8918a9f6a86fbf42bcb9500d83cb372"
KV_CANDIDATES = [Path("/Users/codiefreeman/Zotero/storage/MMANS4Y5/Krogh and Vedelsby - Neural "
                      "Network Ensembles, Cross Validation, and Active Learning.pdf")]
KV_QUOTES = {
    "ambiguity_defined_on_unlabeled_data": (
        "The ambiguity is defined as the variation of the output of ensemble members averaged over "
        "unlabeled data, so it quantifies the disagreement among the networks."),
    "only_useful_if_they_disagree": (
        "A combination of the output of several networks (or other predictors) is only useful if "
        "they disagree on some inputs."),
    "variance_about_the_weighted_mean": (
        "It is simply the variance of the weighted ensemble around the weighed mean, and it "
        "measures the disagreement among the networks on input x."),
    "constraint_on_the_sum_is_crucial": (
        "constrain the weights to be positive and sum to one. The constraint on the sum is "
        "crucial for some of the following results."),
}


def _norm(s):
    for a, b in [("−", "-"), ("–", "-"), ("—", "-"), ("×", "x"),
                 ("’", "'"), ("‘", "'"), ("“", '"'), ("”", '"'),
                 ("ﬁ", "fi"), ("ﬂ", "fl"), (" ", " ")]:
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s).strip()


kv_pdf = next((p for p in KV_CANDIDATES if p.exists()), None)
kv_report = {"checked_at_utc": datetime.now(timezone.utc).isoformat(), "quotes": KV_QUOTES,
             "citation": "Krogh & Vedelsby, NIPS 1994, Neural Network Ensembles, Cross Validation, "
                         "and Active Learning"}
if kv_pdf is None or shutil.which("pdftotext") is None:
    kv_report["status"] = "NOT RE-CHECKED"
    print("*** quotations NOT re-checked (PDF or pdftotext unavailable); verified at build time.")
else:
    kv_report["sha256"] = hashlib.sha256(kv_pdf.read_bytes()).hexdigest()
    body = _norm(subprocess.run(["pdftotext", str(kv_pdf), "-"], capture_output=True, text=True,
                                check=True).stdout)
    found = {k: (_norm(q) in body) for k, q in KV_QUOTES.items()}
    kv_report["quote_found"] = found
    print(f"Krogh & Vedelsby PDF sha256 {kv_report['sha256'][:16]}...")
    for k, ok in found.items():
        print(f"  {'FOUND verbatim' if ok else '*** NOT FOUND':18s}  {k}")
    kv_report["status"] = "VERIFIED" if all(found.values()) else "PARTIAL"
    assert all(found.values()), "a quoted Krogh & Vedelsby sentence is not in the PDF -- fix the text"
    print("\nAll quotations verified. Their definitions, not a paraphrase, are what is implemented below.")
(OUT / "kv_quote_verification.json").write_text(json.dumps(kv_report, indent=2))

Krogh & Vedelsby PDF sha256 88de32e9c1d33849...
  FOUND verbatim      ambiguity_defined_on_unlabeled_data
  FOUND verbatim      only_useful_if_they_disagree
  FOUND verbatim      variance_about_the_weighted_mean
  FOUND verbatim      constraint_on_the_sum_is_crucial

All quotations verified. Their definitions, not a paraphrase, are what is implemented below.


1210

In [3]:
# ---- (2) notebook 37's blend, reproduced from its own saved OOF tables.
OOF37 = {}
for iso in ISOFORMS:
    OOF37[iso] = pd.read_csv(NB37_OUT / "oof_long" / f"oof_long_{iso}_25fold.csv")
ARMS = [c for c in OOF37["CYP3A4"].columns
        if c not in ("repeat", "fold", "inchikey", "Molecule_Name", "y_true", "conf_low", "conf_high")]
assert len(ARMS) == 11, f"expected 11 arms, got {len(ARMS)}"
print(f"notebook 37's 25-fold OOF tables: {len(ARMS)} arms, "
      + ", ".join(f"{i} {len(OOF37[i])} rows" for i in ISOFORMS))


def greedy_blend(df, arm_cols, max_iter=50):
    # Reproduced from notebook 37's own greedy_blend (space="raw" branch): greedy selection with
    # replacement, initialised at the best single arm, on a weighted mean of raw predictions.
    y = df["y_true"].to_numpy(float)
    lo, hi = df["conf_low"].to_numpy(float), df["conf_high"].to_numpy(float)
    P = {a: df[a].to_numpy(float) for a in arm_cols}

    def score_of(counts):
        tot = sum(counts.values())
        v = sum(P[a] * n for a, n in counts.items()) / tot
        return strae(y, v, lo, hi)

    best_single = min(arm_cols, key=lambda a: score_of({a: 1}))
    counts = {best_single: 1}
    best = score_of(counts)
    for _ in range(max_iter - 1):
        cand, cand_s = None, best
        for a in arm_cols:
            trial = dict(counts)
            trial[a] = trial.get(a, 0) + 1
            s = score_of(trial)
            if s < cand_s - 1e-12:
                cand, cand_s = a, s
        if cand is None:
            break
        counts[cand] = counts.get(cand, 0) + 1
        best = cand_s
    tot = sum(counts.values())
    return {a: n / tot for a, n in counts.items()}, best


nb37_blend = pd.read_csv(NB37_OUT / "blend_results.csv")
BLEND_W, repro = {}, []
for iso in ISOFORMS:
    w, s = greedy_blend(OOF37[iso], ARMS)
    BLEND_W[iso] = w
    row = nb37_blend[nb37_blend.isoform == iso].iloc[0]
    want_w = json.loads(row["raw_weights"])
    same = (set(w) == set(want_w)
            and all(abs(w[k] - want_w[k]) < 1e-4 for k in w))
    repro.append({"isoform": iso, "in_sample_here": s, "in_sample_nb37": float(row["raw_in_sample"]),
                  "diff": s - float(row["raw_in_sample"]), "weights_match": same,
                  "weights": json.dumps({k: round(v, 4) for k, v in sorted(w.items(), key=lambda kv: -kv[1])})})
repro = pd.DataFrame(repro)
print("\nreproduction of notebook 37's unconstrained blend:")
print(repro.round(6).to_string(index=False))
assert repro["weights_match"].all(), "member sets differ from notebook 37's"
assert repro["diff"].abs().max() < 1e-9, "in-sample scores differ from notebook 37's"
print("\nEXACT MATCH on member sets and in-sample scores -- Part B is comparable to notebook 37.")

# Krogh & Vedelsby require positive weights summing to one. Checked, not assumed.
for iso, w in BLEND_W.items():
    assert all(v > 0 for v in w.values()) and abs(sum(w.values()) - 1) < 1e-9, iso
print("weights are positive and sum to one on every isoform, as the identity requires.")

notebook 37's 25-fold OOF tables: 11 arms, CYP1A2 7060 rows, CYP2C9 6425 rows, CYP2D6 7465 rows, CYP3A4 11675 rows

reproduction of notebook 37's unconstrained blend:
isoform  in_sample_here  in_sample_nb37  diff  weights_match                                                      weights
 CYP1A2        0.737673        0.737673   0.0           True {"deadzone": 0.5, "aid": 0.3333, "ecfp4_narrow__rf": 0.1667}
 CYP2C9        0.542395        0.542395   0.0           True  {"deadzone": 0.625, "aid": 0.25, "ecfp4_narrow__rf": 0.125}
 CYP2D6        0.871702        0.871702   0.0           True                                            {"deadzone": 1.0}
 CYP3A4        0.456120        0.456120   0.0           True {"deadzone": 0.6, "aid": 0.2, "ecfp4_narrow__lightgbm": 0.2}

EXACT MATCH on member sets and in-sample scores -- Part B is comparable to notebook 37.
weights are positive and sum to one on every isoform, as the identity requires.


### A.1 — Which members actually have blind-set predictions?

The ambiguity on the blind 750 can only be computed for members that were retrained on the full
data and run on the blind set. This is an inventory, not an assumption, and where a member is
missing it is **excluded and named** rather than substituted.

In [4]:
blind_df = pd.read_csv(BLIND)
curated = pd.read_csv(CURATED)
# Keyed on Molecule_Name, not inchikey: NB30's file is a SUBMISSION (SMILES + Molecule_Name, no
# inchikey column), while notebook 12's per-config files carry both. Molecule_Name is the one key
# present in every source and in the curated blind set, and it is unique there -- asserted.
BLIND_KEY = blind_df["Molecule_Name"].tolist()
assert blind_df["Molecule_Name"].is_unique and len(BLIND_KEY) == 750
print(f"blind set: {len(blind_df)} compounds, keyed on Molecule_Name")

# Full-data blind predictions, by the arm name notebook 37 uses.
BLIND_SOURCES = {
    "plain": (NB12_OUT / "blind_predictions" / "chemprop_chemeleoninit.csv",
              "notebook 12's full-data retrain of chemprop_chemeleoninit"),
    "aid": (NB30_OUT / "submission_candidate.csv",
            "NB30 -- the AID auxiliary-head model's full-data retrain, raw and uncorrected"),
    "ecfp4_narrow__rf": (NB12_OUT / "blind_predictions" / "ecfp4_narrow__rf.csv", "notebook 12"),
    "ecfp4_narrow__xgboost": (NB12_OUT / "blind_predictions" / "ecfp4_narrow__xgboost.csv", "notebook 12"),
    "ecfp4_narrow__lightgbm": (NB12_OUT / "blind_predictions" / "ecfp4_narrow__lightgbm.csv", "notebook 12"),
    "mordred_pca__rf": (NB12_OUT / "blind_predictions" / "mordred_pca__rf.csv", "notebook 12"),
    "mordred_pca__xgboost": (NB12_OUT / "blind_predictions" / "mordred_pca__xgboost.csv", "notebook 12"),
    "mordred_pca__lightgbm": (NB12_OUT / "blind_predictions" / "mordred_pca__lightgbm.csv", "notebook 12"),
}
AVAILABLE = {k: v for k, v in BLIND_SOURCES.items() if v[0].exists()}
MISSING = sorted(set(ARMS) - set(AVAILABLE))

inv = []
for a in ARMS:
    has = a in AVAILABLE
    inv.append({"arm": a, "blind_predictions": has,
                "source": AVAILABLE[a][1] if has else "NONE -- never retrained full-data on its own"})
inv = pd.DataFrame(inv)
inv.to_csv(OUT / "blind_member_availability.csv", index=False)
print("\n=== member availability on the blind 750 ===")
print(inv.to_string(index=False))
print(f"\nMISSING: {MISSING}")
print("""
Why each missing arm is missing, checked rather than assumed:
  deadzone  -- notebook 31 screened it on ONE fold; notebook 32's full-data retrain combined the
               dead-zone target with the AID ARCHITECTURE, which is a different model. There is no
               full-data plain+dead-zone model anywhere in outputs/, so it is excluded, not
               substituted with notebook 32's.
  mae_only  -- same: a screen arm only (notebooks 31/35), never retrained full-data.
  single    -- four single-task models, screen-only, never retrained full-data.
""")

# What that leaves of each isoform's blend.
left = []
for iso in ISOFORMS:
    w = BLEND_W[iso]
    keep = {a: v for a, v in w.items() if a in AVAILABLE}
    left.append({"isoform": iso, "n_members": len(w),
                 "n_with_blind_preds": len(keep),
                 "weight_available": sum(keep.values()),
                 "weight_missing": 1 - sum(keep.values()),
                 "members_kept": ";".join(sorted(keep)) or "(none)",
                 "members_dropped": ";".join(sorted(set(w) - set(keep)))})
left = pd.DataFrame(left)
left.to_csv(OUT / "blend_members_on_blind.csv", index=False)
print("=== what that leaves of notebook 37's blend ===")
print(left.to_string(index=False))
print(f"""
READ THIS BEFORE THE NEXT SECTION. `deadzone` carries the MAJORITY of the weight on every isoform
(0.50-1.00), and it is the one member with no blind predictions. So the blind-side ambiguity below
is NOT notebook 37's blend's ambiguity -- it is the ambiguity of whatever subset survives, which on
CYP2D6 is nothing at all. That is reported as the finding it is, not worked around.
""")

blind set: 750 compounds, keyed on Molecule_Name

=== member availability on the blind 750 ===
                   arm  blind_predictions                                                                        source
                 plain               True                     notebook 12's full-data retrain of chemprop_chemeleoninit
      ecfp4_narrow__rf               True                                                                   notebook 12
 ecfp4_narrow__xgboost               True                                                                   notebook 12
ecfp4_narrow__lightgbm               True                                                                   notebook 12
       mordred_pca__rf               True                                                                   notebook 12
  mordred_pca__xgboost               True                                                                   notebook 12
 mordred_pca__lightgbm               True                        

### A.2 — Track 1: ambiguity of notebook 37's blend members, on the subset that exists

`A` is computed exactly as Krogh & Vedelsby define it — `A = Σ_a w_a · mean_x[(V_a(x) − V̄(x))²]`,
the weighted variance of member predictions about the weighted ensemble mean, with weights
renormalised to sum to one over the surviving members (their constraint).

Out-of-fold, `A` is computed **per repeat** and reported as mean ± SD across the five repeats, so
the single blind value can be judged against a noise band rather than a point.

**An asymmetry that cannot be removed and is stated instead**: out-of-fold predictions come from
per-fold models trained on ~80% of the data; blind predictions come from full-data models. `A_oof`
and `A_blind` therefore describe disagreement among *differently trained* instances of the same
recipes. Full-data models are trained on more data and on identical data to each other, which if
anything should *reduce* their disagreement — so a drop from `A_oof` to `A_blind` is partly
expected and is not on its own evidence of collapse.

**Squared error only.** `A` here decomposes squared error, not ST-RAE.

In [5]:
def ambiguity(pred_matrix, weights):
    # Krogh & Vedelsby eq. (2): a(x) = sum_a w_a (V_a(x) - Vbar(x))^2, averaged over x.
    # pred_matrix: dict arm -> 1-D array. weights: dict arm -> w, positive, summing to one.
    arms = list(weights)
    tot = sum(weights[a] for a in arms)
    w = {a: weights[a] / tot for a in arms}
    vbar = sum(w[a] * pred_matrix[a] for a in arms)
    a_x = sum(w[a] * (pred_matrix[a] - vbar) ** 2 for a in arms)
    return float(np.mean(a_x)), vbar


def ebar_and_e(pred_matrix, weights, y):
    arms = list(weights)
    tot = sum(weights[a] for a in arms)
    w = {a: weights[a] / tot for a in arms}
    vbar = sum(w[a] * pred_matrix[a] for a in arms)
    ebar = float(np.mean(sum(w[a] * (y - pred_matrix[a]) ** 2 for a in arms)))
    e = float(np.mean((y - vbar) ** 2))
    return ebar, e


# ---- blind predictions, loaded once and aligned on inchikey.
BLIND_PRED = {}
for a, (path, _) in AVAILABLE.items():
    d = pd.read_csv(path)
    assert len(d) == 750, f"{a}: {len(d)} rows"
    assert set(d["Molecule_Name"]) == set(BLIND_KEY), f"{a}: compound set differs from the blind 750"
    BLIND_PRED[a] = d.set_index("Molecule_Name")
print(f"loaded blind predictions for {len(BLIND_PRED)} arms")

rows = []
for iso in ISOFORMS:
    ep = PIC50_COL[iso]
    w_full = BLEND_W[iso]
    keep = {a: v for a, v in w_full.items() if a in BLIND_PRED}
    if not keep:
        rows.append({"isoform": iso, "n_members_used": 0, "weight_covered": 0.0,
                     "A_oof_mean": np.nan, "A_oof_sd": np.nan, "A_blind": np.nan,
                     "ratio": np.nan, "members": "(none available)"})
        continue
    d = OOF37[iso]
    a_per_repeat = []
    for r in sorted(d["repeat"].unique()):
        sub = d[d["repeat"] == r]
        pm = {a: sub[a].to_numpy(float) for a in keep}
        a_val, _ = ambiguity(pm, keep)
        a_per_repeat.append(a_val)
    pm_b = {a: BLIND_PRED[a].loc[BLIND_KEY, ep].to_numpy(float) for a in keep}
    a_blind, _ = ambiguity(pm_b, keep)
    rows.append({"isoform": iso, "n_members_used": len(keep),
                 "weight_covered": sum(keep.values()),
                 "A_oof_mean": float(np.mean(a_per_repeat)),
                 "A_oof_sd": float(np.std(a_per_repeat, ddof=1)),
                 "A_blind": a_blind,
                 "ratio": a_blind / float(np.mean(a_per_repeat)),
                 "members": ";".join(sorted(keep))})
trackA = pd.DataFrame(rows)
trackA.to_csv(OUT / "ambiguity_track1_blend_members.csv", index=False)
print("\n=== TRACK 1: ambiguity of notebook 37's blend members (available subset only) ===")
print("(squared-error diagnostic; NOT a decomposition of ST-RAE)")
print(trackA.round(4).to_string(index=False))
print(f"""
Every row above uses only {trackA['n_members_used'].max()} of 3 members and covers at most
{trackA['weight_covered'].max():.2f} of the blend's weight, because `deadzone` is missing. CYP2D6 has no
usable row at all. Track 1 therefore cannot answer the question on its own, which is why A.3 runs
the same computation on a pool where every member has both sides.
""")

loaded blind predictions for 8 arms

=== TRACK 1: ambiguity of notebook 37's blend members (available subset only) ===
(squared-error diagnostic; NOT a decomposition of ST-RAE)
isoform  n_members_used  weight_covered  A_oof_mean  A_oof_sd  A_blind  ratio                    members
 CYP1A2               2           0.500      0.0654    0.0099   0.0561 0.8572       aid;ecfp4_narrow__rf
 CYP2C9               2           0.375      0.0400    0.0047   0.0466 1.1631       aid;ecfp4_narrow__rf
 CYP2D6               0           0.000         NaN       NaN      NaN    NaN           (none available)
 CYP3A4               2           0.400      0.0773    0.0029   0.0708 0.9159 aid;ecfp4_narrow__lightgbm

Every row above uses only 2 of 3 members and covers at most
0.50 of the blend's weight, because `deadzone` is missing. CYP2D6 has no
usable row at all. Track 1 therefore cannot answer the question on its own, which is why A.3 runs
the same computation on a pool where every member has both sides.


### A.3 — Track 2: the eleven-config pool, where every member has both sides

This is the comparison Track 1 could not make. All eleven configs from notebook `05`'s pool have
**both** out-of-fold predictions (`outputs/11_caruana_prep/`, the frozen random partition) **and**
full-data blind predictions (`outputs/12_caruana_retrain_predict/blind_predictions/`), with no
member missing and nothing substituted. It is also the pool notebook `24` characterised and the one
notebook `08` selected its ensembles from, so A.4 follows directly from it.

Two weightings are reported: **equal weights** over all eleven (the pool's intrinsic disagreement)
and notebook `08`'s own adopted per-isoform ensembles (A.4).

In [6]:
POOL_OOF = {iso: pd.read_csv(NB11_OUT / f"oof_long_{iso}.csv") for iso in ISOFORMS}
POOL_CFG = [c for c in POOL_OOF["CYP3A4"].columns
            if c not in ("repeat", "fold", "inchikey", "Molecule_Name", "y_true", "conf_low", "conf_high")]
POOL_BLIND = {}
for c in POOL_CFG:
    p = NB12_OUT / "blind_predictions" / f"{c}.csv"
    assert p.exists(), f"no blind predictions for pool config {c}"
    _d = pd.read_csv(p)
    assert set(_d["Molecule_Name"]) == set(BLIND_KEY), f"{c}: compound set differs from the blind 750"
    POOL_BLIND[c] = _d.set_index("Molecule_Name")

# NOT every config has every endpoint on the blind side: notebook 12 saved only the columns its
# own Caruana ensembles needed. Checked rather than assumed, and the pool is therefore built
# PER ISOFORM from the configs that actually carry that endpoint. The out-of-fold side is
# restricted to the same set, so each isoform's comparison stays like-for-like.
POOL_CFG_ISO = {iso: [c for c in POOL_CFG if PIC50_COL[iso] in POOL_BLIND[c].columns]
                for iso in ISOFORMS}
print(f"pool: {len(POOL_CFG)} configs with OOF predictions; blind coverage is per-endpoint:")
for iso in ISOFORMS:
    miss = sorted(set(POOL_CFG) - set(POOL_CFG_ISO[iso]))
    print(f"  {iso}: {len(POOL_CFG_ISO[iso])}/{len(POOL_CFG)} configs"
          + (f"   missing: {miss}" if miss else "   (complete)"))
print("  -> mordred_pca__rf and mordred_pca__xgboost are partial because notebook 12 only wrote")
print("     the endpoints its own ensembles used. Excluded per isoform, never substituted.")

rows = []
for iso in ISOFORMS:
    ep = PIC50_COL[iso]
    cfgs = POOL_CFG_ISO[iso]
    w = {c: 1.0 / len(cfgs) for c in cfgs}
    d = POOL_OOF[iso]
    per_rep = []
    ebar_e = []
    for r in sorted(d["repeat"].unique()):
        sub = d[d["repeat"] == r]
        pm = {c: sub[c].to_numpy(float) for c in cfgs}
        a_val, _ = ambiguity(pm, w)
        per_rep.append(a_val)
        ebar, e = ebar_and_e(pm, w, sub["y_true"].to_numpy(float))
        ebar_e.append((ebar, e, a_val))
    pm_b = {c: POOL_BLIND[c].loc[BLIND_KEY, ep].to_numpy(float) for c in cfgs}
    a_blind, _ = ambiguity(pm_b, w)
    # identity check on the OOF side, where labels exist: E must equal Ebar - A exactly.
    ident = max(abs((eb - a) - e) for eb, e, a in ebar_e)
    rows.append({"isoform": iso, "n_configs": len(cfgs), "A_oof_mean": float(np.mean(per_rep)),
                 "A_oof_sd": float(np.std(per_rep, ddof=1)), "A_blind": a_blind,
                 "ratio_blind_over_oof": a_blind / float(np.mean(per_rep)),
                 "Ebar_oof": float(np.mean([x[0] for x in ebar_e])),
                 "E_oof": float(np.mean([x[1] for x in ebar_e])),
                 "identity_max_abs_err": ident})
pool = pd.DataFrame(rows)
pool.to_csv(OUT / "ambiguity_track2_pool.csv", index=False)
print("\n=== TRACK 2: eleven-config pool, equal weights ===")
print("(squared-error diagnostic; NOT a decomposition of ST-RAE)")
print(pool.round(5).to_string(index=False))
print(f"\nKrogh & Vedelsby identity E = Ebar - A, checked on the OOF side where labels exist:")
print(f"  max abs error over all isoforms and repeats: {pool['identity_max_abs_err'].max():.2e}")
assert pool["identity_max_abs_err"].max() < 1e-9, "the identity does not hold -- implementation error"
print("  -> holds to machine precision, confirming the implementation is theirs and not an approximation.")

print("\n--- does disagreement survive? ---")
for _, r in pool.iterrows():
    band_lo = r["A_oof_mean"] - 2 * r["A_oof_sd"]
    band_hi = r["A_oof_mean"] + 2 * r["A_oof_sd"]
    inside = band_lo <= r["A_blind"] <= band_hi
    verdict = ("WITHIN the OOF repeat-to-repeat band" if inside
               else ("ABOVE it" if r["A_blind"] > band_hi else "BELOW it"))
    print(f"  {r['isoform']}: A_oof {r['A_oof_mean']:.4f}+/-{r['A_oof_sd']:.4f}, "
          f"A_blind {r['A_blind']:.4f} ({r['ratio_blind_over_oof']:.2f}x) -- {verdict}")

pool: 11 configs with OOF predictions; blind coverage is per-endpoint:
  CYP1A2: 10/11 configs   missing: ['mordred_pca__rf']
  CYP2C9: 9/11 configs   missing: ['mordred_pca__rf', 'mordred_pca__xgboost']
  CYP2D6: 11/11 configs   (complete)
  CYP3A4: 9/11 configs   missing: ['mordred_pca__rf', 'mordred_pca__xgboost']
  -> mordred_pca__rf and mordred_pca__xgboost are partial because notebook 12 only wrote
     the endpoints its own ensembles used. Excluded per isoform, never substituted.

=== TRACK 2: eleven-config pool, equal weights ===
(squared-error diagnostic; NOT a decomposition of ST-RAE)
isoform  n_configs  A_oof_mean  A_oof_sd  A_blind  ratio_blind_over_oof  Ebar_oof   E_oof  identity_max_abs_err
 CYP1A2         10     0.09849   0.00684  0.13336               1.35402   0.90358 0.80509                   0.0
 CYP2C9          9     0.06508   0.00301  0.09784               1.50350   0.45345 0.38838                   0.0
 CYP2D6         11     0.06545   0.00417  0.08384             

In [7]:
# ---- A.4: the retrospective check on notebook 08's adopted ensembles, which lost 0.116 macro.
nb08 = pd.read_csv(NB08_OUT / "step3_summary.csv")
print("notebook 08's adopted ensembles, read from its own step3_summary.csv:")
print(nb08[["isoform", "decision", "winning_combo"]].to_string(index=False))

# NB10 submitted these, and kept its OWN per-member full-data blind predictions -- including the
# notebook 06/07 variants its CYP2D6 ensemble actually used. Those are the correct blind inputs.
# NB10 saved only the endpoints each member was actually used for, so these files are
# per-isoform rather than all-four. Checked below rather than assumed.
NB10_BLIND = {p.stem: pd.read_csv(p).set_index("Molecule_Name")
              for p in sorted((NB10_OUT / "blind_predictions").glob("*.csv"))}
print("\nendpoints present in each NB10 member file:")
for k, v in NB10_BLIND.items():
    print(f"  {k:52s} {[c for c in v.columns if 'pIC50' in c]}")
print(f"\nNB10's own full-data member predictions: {sorted(NB10_BLIND)}")

# Map notebook 08's combo member names onto NB10's files. The mapping is explicit, not guessed:
# CYP2D6's ensemble used the 06-excluded / 07-tuned variants, every other isoform the unexcluded.
NB08_MEMBER_FILE = {
    ("CYP1A2", "chemprop_chemeleoninit"): "chemprop_chemeleoninit__unexcluded",
    ("CYP1A2", "ecfp4_narrow__rf"): "ecfp4_narrow__rf__unexcluded",
    ("CYP1A2", "chemeleon__rf"): "chemeleon__rf__unexcluded",
    ("CYP1A2", "ecfp4_narrow__lightgbm"): "ecfp4_narrow__lightgbm__unexcluded",
    ("CYP2D6", "chemeleon__rf"): "chemeleon__rf__cyp2d6_residual_excluded_tuned",
    ("CYP2D6", "chemprop_chemeleoninit"): "chemprop_chemeleoninit__cyp2d6_residual_excluded",
    ("CYP3A4", "chemprop_chemeleoninit"): "chemprop_chemeleoninit__unexcluded",
    ("CYP3A4", "ecfp4_narrow__lightgbm"): "ecfp4_narrow__lightgbm__unexcluded",
}

rows = []
for _, r in nb08.iterrows():
    iso = r["isoform"]
    if not isinstance(r["winning_combo"], str) or not r["winning_combo"]:
        rows.append({"isoform": iso, "n_members": 1, "A_oof_mean": np.nan, "A_oof_sd": np.nan,
                     "A_blind": np.nan, "ratio_blind_over_oof": np.nan,
                     "note": "no ensemble adopted -- singleton winner, ambiguity undefined"})
        continue
    members = r["winning_combo"].split("+")
    w = {m: 1.0 / len(members) for m in members}        # notebook 08 used SIMPLE averaging
    ep = PIC50_COL[iso]
    d = POOL_OOF[iso]
    per_rep = []
    for rep in sorted(d["repeat"].unique()):
        sub = d[d["repeat"] == rep]
        pm = {m: sub[m].to_numpy(float) for m in members}
        per_rep.append(ambiguity(pm, w)[0])
    files = [NB08_MEMBER_FILE.get((iso, m)) for m in members]
    assert all(f in NB10_BLIND for f in files), f"{iso}: unmapped member among {members}"
    missing_ep = [f for f in files if ep not in NB10_BLIND[f].columns]
    if missing_ep:
        rows.append({"isoform": iso, "n_members": len(members), "A_oof_mean": np.nan,
                     "A_oof_sd": np.nan, "A_blind": np.nan, "ratio_blind_over_oof": np.nan,
                     "note": f"NB10 saved no {ep} column for {missing_ep} -- not computable"})
        continue
    pm_b = {m: NB10_BLIND[f].loc[BLIND_KEY, ep].to_numpy(float) for m, f in zip(members, files)}
    a_blind = ambiguity(pm_b, w)[0]
    rows.append({"isoform": iso, "n_members": len(members),
                 "A_oof_mean": float(np.mean(per_rep)),
                 "A_oof_sd": float(np.std(per_rep, ddof=1)), "A_blind": a_blind,
                 "ratio_blind_over_oof": a_blind / float(np.mean(per_rep)),
                 "note": r["winning_combo"]})
nb08_amb = pd.DataFrame(rows)
nb08_amb.to_csv(OUT / "ambiguity_nb08_ensembles.csv", index=False)
print("\n=== A.4: notebook 08's own ensembles, the ones NB10 submitted and lost 0.116 macro with ===")
print("(squared-error diagnostic; OOF side on the random partition 08 selected on)")
print(nb08_amb.round(5).to_string(index=False))
print("""
Note on the OOF side for CYP2D6: notebook 08's CYP2D6 ensemble used notebook 06/07's excluded and
tuned variants, whose BLIND predictions NB10 saved separately (used above). Its OOF columns here are
the pool's plain unexcluded ones, because that is what outputs/11_caruana_prep/ holds. The CYP2D6
OOF ambiguity is therefore an approximation to what 08 itself saw; the blind side is exact. Flagged
rather than silently mixed.
""")

notebook 08's adopted ensembles, read from its own step3_summary.csv:
isoform                                 decision                                                                winning_combo
 CYP1A2                         ensemble adopted chemprop_chemeleoninit+ecfp4_narrow__rf+chemeleon__rf+ecfp4_narrow__lightgbm
 CYP2C9 single model stands (no ensemble tested)                                                                          NaN
 CYP2D6                         ensemble adopted                                         chemeleon__rf+chemprop_chemeleoninit
 CYP3A4                         ensemble adopted                                chemprop_chemeleoninit+ecfp4_narrow__lightgbm



endpoints present in each NB10 member file:
  chemeleon__rf__cyp2d6_residual_excluded_tuned        ['CYP2D6_pIC50_direct_inhibition']
  chemeleon__rf__unexcluded                            ['CYP1A2_pIC50_direct_inhibition']
  chemprop_chemeleoninit__cyp2d6_residual_excluded     ['CYP1A2_pIC50_direct_inhibition', 'CYP2C9_pIC50_direct_inhibition', 'CYP2D6_pIC50_direct_inhibition', 'CYP3A4_pIC50_direct_inhibition']
  chemprop_chemeleoninit__unexcluded                   ['CYP1A2_pIC50_direct_inhibition', 'CYP2C9_pIC50_direct_inhibition', 'CYP2D6_pIC50_direct_inhibition', 'CYP3A4_pIC50_direct_inhibition']
  ecfp4_narrow__lightgbm__unexcluded                   ['CYP1A2_pIC50_direct_inhibition', 'CYP3A4_pIC50_direct_inhibition']
  ecfp4_narrow__rf__unexcluded                         ['CYP1A2_pIC50_direct_inhibition']

NB10's own full-data member predictions: ['chemeleon__rf__cyp2d6_residual_excluded_tuned', 'chemeleon__rf__unexcluded', 'chemprop_chemeleoninit__cyp2d6_residual_excluded', 'che

## Part B — what does a shared member set cost?

Three levels of constraint, each scored the same way so the costs are separable:

| level | member set | weights | what a submission built this way is |
|---|---|---|---|
| **unconstrained** | per isoform | per isoform | four independent ensembles |
| **shared members** | one set, all isoforms | per isoform | one member set, four weightings |
| **shared members + weights** | one set | one weighting | genuinely one object |

**The honesty protocol is notebook `37`'s, unchanged**: for each held-out fold, everything is fitted
on the other 24 and scored on the held-out one, rotating over all 25. For the constrained levels
that includes **re-selecting the shared member set inside each fold**, not just refitting weights —
otherwise the constrained figure would carry an advantage the unconstrained one does not, and the
two would not be comparable, which is the whole point of the exercise.

The shared set is chosen by **forward selection**: start empty, add whichever arm most improves the
macro objective, stop when nothing improves it. That is a search over sets rather than over the
2,048 subsets exhaustively — a decision recorded below, taken because the nested protocol needs the
search run 25 times over and because forward selection is the same mechanic notebook `37`'s greedy
already uses.

**Macro objective**: the unweighted mean of the four isoform ST-RAEs, which is how the leaderboard
computes its MA row (verified in `docs/leaderboard_submissions.md`'s own provenance section:
every published macro equals the mean of its four isoform values).

In [8]:
# ---- shared-set machinery. All three levels share one scoring path so they are comparable.
def iso_arrays(df, arm_cols):
    return ({a: df[a].to_numpy(float) for a in arm_cols},
            df["y_true"].to_numpy(float), df["conf_low"].to_numpy(float),
            df["conf_high"].to_numpy(float))


def fit_weights_restricted(df, allowed, max_iter=50):
    # Notebook 37's greedy, restricted to `allowed`. Returns (weights, in-sample ST-RAE).
    return greedy_blend(df, list(allowed), max_iter=max_iter)


def score_with_weights(df, weights):
    P, y, lo, hi = iso_arrays(df, list(weights))
    v = sum(weights[a] * P[a] for a in weights)
    return strae(y, v, lo, hi)


def fit_shared_weights(dfs, allowed, max_iter=50):
    # ONE weight vector across all four isoforms, chosen by greedy selection with replacement on
    # the macro objective (mean of the four isoform ST-RAEs).
    cache = {iso: iso_arrays(dfs[iso], list(allowed)) for iso in dfs}

    def macro_of(counts):
        tot = sum(counts.values())
        vals = []
        for iso in dfs:
            P, y, lo, hi = cache[iso]
            v = sum(P[a] * n for a, n in counts.items()) / tot
            vals.append(strae(y, v, lo, hi))
        return float(np.mean(vals))

    best1 = min(allowed, key=lambda a: macro_of({a: 1}))
    counts, best = {best1: 1}, macro_of({best1: 1})
    for _ in range(max_iter - 1):
        cand, cand_s = None, best
        for a in allowed:
            t = dict(counts)
            t[a] = t.get(a, 0) + 1
            s = macro_of(t)
            if s < cand_s - 1e-12:
                cand, cand_s = a, s
        if cand is None:
            break
        counts[cand] = counts.get(cand, 0) + 1
        best = cand_s
    tot = sum(counts.values())
    return {a: n / tot for a, n in counts.items()}, best


def forward_shared_set(dfs, arms, shared_weights=False, max_size=None):
    # Forward selection over shared MEMBER SETS, on the macro objective.
    chosen, best_macro = [], np.inf
    cap = max_size or len(arms)
    while len(chosen) < cap:
        cand, cand_macro, cand_payload = None, best_macro, None
        for a in arms:
            if a in chosen:
                continue
            trial = chosen + [a]
            if shared_weights:
                w, m = fit_shared_weights(dfs, trial)
                payload = w
            else:
                ws, vals = {}, []
                for iso in dfs:
                    w_i, s_i = fit_weights_restricted(dfs[iso], trial)
                    ws[iso] = w_i
                    vals.append(s_i)
                m, payload = float(np.mean(vals)), ws
            if m < cand_macro - 1e-12:
                cand, cand_macro, cand_payload = a, m, payload
        if cand is None:
            break
        chosen.append(cand)
        best_macro, best_payload = cand_macro, cand_payload
    return chosen, best_macro, best_payload


print("shared-set machinery ready.")

shared-set machinery ready.


In [9]:
# ---- IN-SAMPLE fits at all three levels, for reference (the nested figures follow).
t0 = time.time()
unc_w = {iso: BLEND_W[iso] for iso in ISOFORMS}
unc_in = {iso: greedy_blend(OOF37[iso], ARMS)[1] for iso in ISOFORMS}

shared_set, shared_macro_in, shared_w_in = forward_shared_set(OOF37, ARMS, shared_weights=False)
shared_both_set, shared_both_macro_in, shared_both_w_in = forward_shared_set(
    OOF37, ARMS, shared_weights=True)
print(f"in-sample fits done in {time.time()-t0:.1f}s\n")

print("UNCONSTRAINED (notebook 37's own):")
for iso in ISOFORMS:
    print(f"  {iso}: {unc_in[iso]:.4f}  {json.dumps({k: round(v,4) for k,v in sorted(unc_w[iso].items(), key=lambda kv:-kv[1])})}")
print(f"  macro {np.mean([unc_in[i] for i in ISOFORMS]):.4f}")

print(f"\nSHARED MEMBERS, per-isoform weights -- set: {shared_set}")
for iso in ISOFORMS:
    print(f"  {iso}: {score_with_weights(OOF37[iso], shared_w_in[iso]):.4f}  "
          f"{json.dumps({k: round(v,4) for k,v in sorted(shared_w_in[iso].items(), key=lambda kv:-kv[1])})}")
print(f"  macro {shared_macro_in:.4f}")

print(f"\nSHARED MEMBERS AND WEIGHTS -- set: {shared_both_set}")
print(f"  weights {json.dumps({k: round(v,4) for k,v in sorted(shared_both_w_in.items(), key=lambda kv:-kv[1])})}")
for iso in ISOFORMS:
    print(f"  {iso}: {score_with_weights(OOF37[iso], shared_both_w_in):.4f}")
print(f"  macro {shared_both_macro_in:.4f}")

in-sample fits done in 0.3s

UNCONSTRAINED (notebook 37's own):
  CYP1A2: 0.7377  {"deadzone": 0.5, "aid": 0.3333, "ecfp4_narrow__rf": 0.1667}
  CYP2C9: 0.5424  {"deadzone": 0.625, "aid": 0.25, "ecfp4_narrow__rf": 0.125}
  CYP2D6: 0.8717  {"deadzone": 1.0}
  CYP3A4: 0.4561  {"deadzone": 0.6, "aid": 0.2, "ecfp4_narrow__lightgbm": 0.2}
  macro 0.6520

SHARED MEMBERS, per-isoform weights -- set: ['deadzone', 'aid', 'ecfp4_narrow__rf', 'ecfp4_narrow__lightgbm']
  CYP1A2: 0.7377  {"deadzone": 0.5, "aid": 0.3333, "ecfp4_narrow__rf": 0.1667}
  CYP2C9: 0.5424  {"deadzone": 0.625, "aid": 0.25, "ecfp4_narrow__rf": 0.125}
  CYP2D6: 0.8717  {"deadzone": 1.0}
  CYP3A4: 0.4561  {"deadzone": 0.6, "aid": 0.2, "ecfp4_narrow__lightgbm": 0.2}
  macro 0.6520

SHARED MEMBERS AND WEIGHTS -- set: ['deadzone', 'aid', 'ecfp4_narrow__xgboost']
  weights {"deadzone": 0.625, "aid": 0.25, "ecfp4_narrow__xgboost": 0.125}
  CYP1A2: 0.7391
  CYP2C9: 0.5444
  CYP2D6: 0.8605
  CYP3A4: 0.4568
  macro 0.6502


In [10]:
# ---- NESTED evaluation at all three levels. Everything -- member set AND weights -- is fitted on
# the other 24 folds and scored on the held-out one, rotating over all 25.
UNITS = sorted(set(map(tuple, OOF37["CYP1A2"][["repeat", "fold"]].values.tolist())))
print(f"{len(UNITS)} (repeat, fold) units\n")
t0 = time.time()

held_pred = {lvl: {iso: [] for iso in ISOFORMS} for lvl in ("unconstrained", "shared_members", "shared_both")}
held_true = {iso: [] for iso in ISOFORMS}
held_lo = {iso: [] for iso in ISOFORMS}
held_hi = {iso: [] for iso in ISOFORMS}
fold_sets = {"shared_members": [], "shared_both": []}

for (r, f) in UNITS:
    fit_dfs = {iso: OOF37[iso][~((OOF37[iso]["repeat"] == r) & (OOF37[iso]["fold"] == f))]
               for iso in ISOFORMS}
    out_dfs = {iso: OOF37[iso][(OOF37[iso]["repeat"] == r) & (OOF37[iso]["fold"] == f)]
               for iso in ISOFORMS}
    for iso in ISOFORMS:
        held_true[iso].append(out_dfs[iso]["y_true"].to_numpy(float))
        held_lo[iso].append(out_dfs[iso]["conf_low"].to_numpy(float))
        held_hi[iso].append(out_dfs[iso]["conf_high"].to_numpy(float))

    # level 1: unconstrained, per-isoform set and weights
    for iso in ISOFORMS:
        w, _ = fit_weights_restricted(fit_dfs[iso], ARMS)
        held_pred["unconstrained"][iso].append(
            sum(w[a] * out_dfs[iso][a].to_numpy(float) for a in w))

    # level 2: one shared set (re-selected here), per-isoform weights
    s_set, _, s_w = forward_shared_set(fit_dfs, ARMS, shared_weights=False)
    fold_sets["shared_members"].append(";".join(s_set))
    for iso in ISOFORMS:
        held_pred["shared_members"][iso].append(
            sum(s_w[iso][a] * out_dfs[iso][a].to_numpy(float) for a in s_w[iso]))

    # level 3: one shared set AND one shared weight vector
    b_set, _, b_w = forward_shared_set(fit_dfs, ARMS, shared_weights=True)
    fold_sets["shared_both"].append(";".join(b_set))
    for iso in ISOFORMS:
        held_pred["shared_both"][iso].append(
            sum(b_w[a] * out_dfs[iso][a].to_numpy(float) for a in b_w))

print(f"nested evaluation done in {time.time()-t0:.1f}s")

nested = {}
for lvl in held_pred:
    nested[lvl] = {}
    for iso in ISOFORMS:
        nested[lvl][iso] = strae(np.concatenate(held_true[iso]),
                                 np.concatenate(held_pred[lvl][iso]),
                                 np.concatenate(held_lo[iso]), np.concatenate(held_hi[iso]))
    nested[lvl]["macro"] = float(np.mean([nested[lvl][i] for i in ISOFORMS]))

# sanity: the unconstrained nested figure must reproduce notebook 37's published one.
for iso in ISOFORMS:
    want = float(nb37_blend[nb37_blend.isoform == iso]["raw_nested"].iloc[0])
    got = nested["unconstrained"][iso]
    assert abs(got - want) < 1e-9, f"{iso}: nested {got:.6f} != notebook 37's {want:.6f}"
print("unconstrained nested figures reproduce notebook 37's exactly on all four isoforms.")

25 (repeat, fold) units



nested evaluation done in 6.1s
unconstrained nested figures reproduce notebook 37's exactly on all four isoforms.


In [11]:
# ---- the cost table.
rows = []
for iso in ISOFORMS + ["macro"]:
    u = nested["unconstrained"][iso]
    s = nested["shared_members"][iso]
    b = nested["shared_both"][iso]
    rows.append({"isoform": iso, "unconstrained": u, "shared_members": s, "shared_both": b,
                 "cost_shared_members": s - u, "cost_shared_both": b - u,
                 "cost_of_shared_weights_on_top": b - s})
cost = pd.DataFrame(rows)
cost.to_csv(OUT / "shared_set_cost.csv", index=False)

plain_nested = {iso: float(nb37_blend[nb37_blend.isoform == iso]["plain_strae"].iloc[0])
                for iso in ISOFORMS}
plain_nested["macro"] = float(np.mean([plain_nested[i] for i in ISOFORMS]))

print("=" * 86)
print("PART B -- NESTED ST-RAE AT THREE LEVELS OF CONSTRAINT (lower is better)")
print("=" * 86)
print(f"\n{'':9s} {'PLAIN':>9s} {'unconstr':>9s} {'shared mem':>11s} {'shared both':>12s} "
      f"{'cost(mem)':>10s} {'cost(both)':>11s}")
for _, r in cost.iterrows():
    print(f"{r['isoform']:9s} {plain_nested[r['isoform']]:9.4f} {r['unconstrained']:9.4f} "
          f"{r['shared_members']:11.4f} {r['shared_both']:12.4f} "
          f"{r['cost_shared_members']:+10.4f} {r['cost_shared_both']:+11.4f}")

from collections import Counter
print(f"\nshared member set chosen in-sample:       {shared_set}")
print(f"shared member set, per-fold agreement:   "
      f"{Counter(fold_sets['shared_members']).most_common(3)}")
print(f"shared set + weights chosen in-sample:   {shared_both_set}")
print(f"  with weights {json.dumps({k: round(v,4) for k,v in sorted(shared_both_w_in.items(), key=lambda kv:-kv[1])})}")
print(f"shared-both set, per-fold agreement:     "
      f"{Counter(fold_sets['shared_both']).most_common(3)}")
pd.DataFrame({"level": ["shared_members"] * len(UNITS) + ["shared_both"] * len(UNITS),
              "repeat": [u[0] for u in UNITS] * 2, "fold": [u[1] for u in UNITS] * 2,
              "set": fold_sets["shared_members"] + fold_sets["shared_both"]}).to_csv(
    OUT / "per_fold_shared_sets.csv", index=False)

PART B -- NESTED ST-RAE AT THREE LEVELS OF CONSTRAINT (lower is better)

              PLAIN  unconstr  shared mem  shared both  cost(mem)  cost(both)
CYP1A2       0.8534    0.7399      0.7400       0.7391    +0.0002     -0.0008
CYP2C9       0.6541    0.5438      0.5438       0.5444    +0.0000     +0.0006
CYP2D6       0.9913    0.8717      0.8717       0.8605    +0.0000     -0.0113
CYP3A4       0.5411    0.4561      0.4570       0.4568    +0.0009     +0.0007
macro        0.7600    0.6529      0.6531       0.6502    +0.0003     -0.0027

shared member set chosen in-sample:       ['deadzone', 'aid', 'ecfp4_narrow__rf', 'ecfp4_narrow__lightgbm']
shared member set, per-fold agreement:   [('deadzone;aid;ecfp4_narrow__rf;ecfp4_narrow__lightgbm', 18), ('deadzone;aid;ecfp4_narrow__rf', 6), ('deadzone;aid;ecfp4_narrow__xgboost;ecfp4_narrow__lightgbm', 1)]
shared set + weights chosen in-sample:   ['deadzone', 'aid', 'ecfp4_narrow__xgboost']
  with weights {"deadzone": 0.625, "aid": 0.25, "ecfp4_n

In [12]:
# ---- CYP2D6 specifically: what does it lose by being made to carry members it rejects?
print("=" * 86)
print("CYP2D6 -- the isoform whose own search rejects every member but one")
print("=" * 86)
own = BLEND_W["CYP2D6"]
print(f"\nits own unconstrained choice: {json.dumps({k: round(v,4) for k,v in own.items()})}")
print(f"  notebook 37 recorded 25/25 per-fold selections agreeing on that single member.")
d6 = cost[cost.isoform == "CYP2D6"].iloc[0]
print(f"\n  unconstrained nested   {d6['unconstrained']:.4f}")
print(f"  shared members         {d6['shared_members']:.4f}   cost {d6['cost_shared_members']:+.4f}")
print(f"  shared members+weights {d6['shared_both']:.4f}   cost {d6['cost_shared_both']:+.4f}")
print(f"  PLAIN (no blend)       {plain_nested['CYP2D6']:.4f}")

others = cost[cost.isoform.isin(["CYP1A2", "CYP2C9", "CYP3A4"])]
print(f"\nthe other three, for comparison:")
for _, r in others.iterrows():
    kept = (r["unconstrained"] - r["shared_members"])
    print(f"  {r['isoform']}: cost of shared members {r['cost_shared_members']:+.4f}  "
          f"(shared-both {r['cost_shared_both']:+.4f})")
print(f"\n  mean cost of the shared-member constraint, other three: "
      f"{others['cost_shared_members'].mean():+.4f}")
print(f"  CYP2D6's own cost:                                    "
      f"{d6['cost_shared_members']:+.4f}")

# WHY it is zero, checked rather than asserted: with weights still fitted per isoform, CYP2D6 can
# simply give the shared members no weight and recover its own unconstrained solution exactly.
w_d6_shared = shared_w_in["CYP2D6"]
print(f"\n  CYP2D6's fitted weights under the shared-member constraint: "
      f"{json.dumps({k: round(v,4) for k,v in w_d6_shared.items()})}")
recovered = (set(w_d6_shared) == set(own)
             and all(abs(w_d6_shared[k] - own[k]) < 1e-9 for k in own))
print(f"  identical to its unconstrained choice: {recovered}")
print(f"""
  THAT is why the cost is exactly zero, and it generalises: a shared member SET costs an isoform
  nothing as long as weights are still fitted per isoform, because any isoform can zero out the
  members it does not want. The brief's framing -- CYP2D6 "forced to carry members its own search
  rejects" -- does not bite at this level of constraint. It only bites at the next one, where one
  weight vector is imposed on all four and CYP2D6 can no longer opt out.""")

# The level that actually constrains CYP2D6.
print(f"\n  At the stricter level -- one weight vector for all four -- CYP2D6 moves "
      f"{d6['cost_shared_both']:+.4f},")
print(f"  which is {'an IMPROVEMENT, not a cost' if d6['cost_shared_both'] < 0 else 'a genuine cost'}. "
      f"The shared vector is")
print(f"  {json.dumps({k: round(v,4) for k,v in sorted(shared_both_w_in.items(), key=lambda kv:-kv[1])})},")
print(f"  so CYP2D6 is made to carry {len(shared_both_w_in)-1} members it would otherwise reject, and still does better")
print(f"  out-of-fold than on its own. A constraint that improves a nested figure is acting as")
print(f"  REGULARISATION: the per-isoform search was overfitting its own folds, and being forced to")
print(f"  share a weight vector with three other isoforms costs it that overfit. Reported as the")
print(f"  finding it is, and tempered by the standing caveat that this is still out-of-fold.")

# Does the shared set still beat PLAIN on every isoform? That is the question that decides whether
# the route is viable at all.
print(f"\n--- does the constrained blend still beat PLAIN (no blend) on every isoform? ---")
for _, r in cost.iterrows():
    if r["isoform"] == "macro":
        continue
    for lvl in ("shared_members", "shared_both"):
        gain = plain_nested[r["isoform"]] - r[lvl]
        print(f"  {r['isoform']:8s} {lvl:15s} {r[lvl]:.4f} vs PLAIN {plain_nested[r['isoform']]:.4f} "
              f"-> {'beats PLAIN by ' + format(gain, '+.4f') if gain > 0 else 'WORSE THAN PLAIN'}")

CYP2D6 -- the isoform whose own search rejects every member but one

its own unconstrained choice: {"deadzone": 1.0}
  notebook 37 recorded 25/25 per-fold selections agreeing on that single member.

  unconstrained nested   0.8717
  shared members         0.8717   cost +0.0000
  shared members+weights 0.8605   cost -0.0113
  PLAIN (no blend)       0.9913

the other three, for comparison:
  CYP1A2: cost of shared members +0.0002  (shared-both -0.0008)
  CYP2C9: cost of shared members +0.0000  (shared-both +0.0006)
  CYP3A4: cost of shared members +0.0009  (shared-both +0.0007)

  mean cost of the shared-member constraint, other three: +0.0003
  CYP2D6's own cost:                                    +0.0000

  CYP2D6's fitted weights under the shared-member constraint: {"deadzone": 1.0}
  identical to its unconstrained choice: True

  THAT is why the cost is exactly zero, and it generalises: a shared member SET costs an isoform
  nothing as long as weights are still fitted per isoform, be

## Part C — Figures

House style, matching notebooks 21/24/25/26/27/29/31/33/36/38: `figsize=(10,6)`, `dpi=150`,
`bbox_inches="tight"`, base font 10, top and right spines removed, legends without frames, titles
11pt bold and left-aligned **stating the finding**, `#8C8C8C` grey for reference series and
`#B0413E` for the series carrying the argument.

In [13]:
plt.rcParams.update({"font.size": 10, "figure.dpi": 110, "savefig.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "legend.frameon": False})
GREY, ACCENT, MID, GOOD = "#8C8C8C", "#B0413E", "#4C4C4C", "#2C7A4B"


def finish(fig, path):
    fig.savefig(path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"wrote {path}")

In [14]:
# ---- FIGURE 1: ambiguity out-of-fold vs blind, the eleven-config pool.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.3),
                               gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.36})
x = np.arange(len(ISOFORMS))
oofm = pool.set_index("isoform").loc[ISOFORMS, "A_oof_mean"].to_numpy()
oofs = pool.set_index("isoform").loc[ISOFORMS, "A_oof_sd"].to_numpy()
bl = pool.set_index("isoform").loc[ISOFORMS, "A_blind"].to_numpy()
ax1.bar(x - 0.19, oofm, yerr=oofs, width=0.36, color=GREY, label="out-of-fold (5 repeats)",
        error_kw={"ecolor": MID, "elinewidth": 1, "capsize": 3})
ax1.bar(x + 0.19, bl, width=0.36, color=ACCENT, label="blind 750 (full-data models)")
ax1.set_xticks(x)
ax1.set_xticklabels(ISOFORMS, fontsize=9)
ax1.set_ylabel("ensemble ambiguity A  (pIC50$^2$)")
ax1.set_title("eleven-config pool, equal weights", fontsize=10)
ax1.legend(fontsize=8)
for xi, (a, b) in enumerate(zip(oofm, bl)):
    ax1.text(xi + 0.19, b, f"{b/a:.2f}x", ha="center", va="bottom", fontsize=8, fontweight="bold",
             color=ACCENT)

rat = pool.set_index("isoform").loc[ISOFORMS, "ratio_blind_over_oof"].to_numpy()
cols = [ACCENT if r < 1 else GOOD for r in rat]
ax2.barh(x, rat, color=cols, height=0.6)
ax2.axvline(1.0, color="black", lw=1.0)
ax2.set_yticks(x)
ax2.set_yticklabels(ISOFORMS, fontsize=9)
ax2.set_xlabel("A(blind) / A(out-of-fold)")
ax2.set_title("1.0 = disagreement fully preserved", fontsize=10)
ax2.invert_yaxis()
_lo, _hi = rat.min(), rat.max()
fig.suptitle(f"Member disagreement on the blind set is {_lo:.2f}-{_hi:.2f}x its out-of-fold level "
             f"— it does not collapse",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.tight_layout(rect=[0, 0, 1, 0.90])
finish(fig, FIG_OUT / "ambiguity_oof_vs_blind.png")

/var/folders/g2/8zyjglrn4c7dqmvyh0hq9p4m0000gn/T/ipykernel_80626/4046009489.py:33: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.90])


wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/39_ambiguity_and_shared_blend/figures/ambiguity_oof_vs_blind.png


**Figure 1 — `ambiguity_oof_vs_blind.png`.** Krogh & Vedelsby's ensemble ambiguity `A` — the
weighted variance of member predictions about the ensemble mean, computable without labels —
measured on the eleven-config pool with equal weights. *Left:* out-of-fold (grey, with the
repeat-to-repeat spread as error bars) beside the blind 750 (red), with the ratio annotated.
*Right:* the same ratio against 1.0, where 1.0 would mean disagreement is exactly as large on the
compounds that get scored as on the compounds the blend was fitted to. This is the pool where every
member has predictions on **both** sides, so nothing is missing and nothing is substituted. **A is
computed on squared error, as the identity requires, and is a diagnostic of disagreement — it does
not decompose ST-RAE.**

In [15]:
# ---- FIGURE 2: the cost of each level of constraint.
fig, ax = plt.subplots(figsize=(10, 4.6))
order = ISOFORMS + ["macro"]
xs = np.arange(len(order))
c = cost.set_index("isoform").loc[order]
pl = [plain_nested[i] for i in order]
ax.plot(xs, pl, "o--", color=MID, lw=1.0, ms=6, label="PLAIN (no blend)", zorder=2)
ax.plot(xs, c["unconstrained"], "o-", color=GOOD, lw=1.6, ms=7,
        label="unconstrained (four ensembles)", zorder=3)
ax.plot(xs, c["shared_members"], "s-", color=ACCENT, lw=1.6, ms=7,
        label="shared members, per-isoform weights", zorder=3)
ax.plot(xs, c["shared_both"], "^-", color=GREY, lw=1.4, ms=7,
        label="shared members AND weights (one object)", zorder=3)
for i, iso in enumerate(order):
    d = c.loc[iso, "cost_shared_members"]
    ax.annotate(f"{d:+.4f}", (i, c.loc[iso, "shared_members"]), textcoords="offset points",
                xytext=(0, 9), ha="center", fontsize=7.5, color=ACCENT, fontweight="bold")
ax.set_xticks(xs)
ax.set_xticklabels([o.upper() if o == "macro" else o for o in order], fontsize=9)
ax.set_ylabel("nested out-of-fold ST-RAE, lower is better")
ax.legend(fontsize=8, loc="upper left", ncol=2)
_d6 = float(c.loc["CYP2D6", "cost_shared_members"])
_oth = float(c.loc[["CYP1A2", "CYP2C9", "CYP3A4"], "cost_shared_members"].mean())
fig.suptitle(f"Sharing one member set costs CYP2D6 {_d6:+.4f} ST-RAE and the other three "
             f"{_oth:+.4f} on average",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.tight_layout(rect=[0, 0, 1, 0.92])
finish(fig, FIG_OUT / "shared_set_cost.png")
cost.to_csv(OUT / "figure2_shared_set_cost.csv", index=False)

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/39_ambiguity_and_shared_blend/figures/shared_set_cost.png


**Figure 2 — `shared_set_cost.png`.** Nested out-of-fold ST-RAE at each level of constraint, one
point per isoform plus the macro, all under notebook `37`'s own rotation protocol (member set *and*
weights fitted on 24 folds, scored on the held-out one, 25 times). The dashed grey line is PLAIN
with no blend at all, so the vertical distance from it to any solid line is what blending buys.
Red annotations give the cost of the shared-member constraint relative to the unconstrained blend.
The gap between the red squares and the grey triangles is the additional cost of also sharing one
weight vector — the step from "one member set, four weightings" to a genuinely single object.

In [16]:
# ---- FIGURE 3: where the ambiguity actually lives -- per-member contribution, pool.
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), sharey=True)
for ax, side in zip(axes, ["out-of-fold", "blind"]):
    mat = []
    for iso in ISOFORMS:
        ep = PIC50_COL[iso]
        cfgs = POOL_CFG_ISO[iso]
        w = {c_: 1.0 / len(cfgs) for c_ in cfgs}
        if side == "out-of-fold":
            d = POOL_OOF[iso]
            pm = {c_: d[c_].to_numpy(float) for c_ in cfgs}
        else:
            pm = {c_: POOL_BLIND[c_].loc[BLIND_KEY, ep].to_numpy(float) for c_ in cfgs}
        _, vbar = ambiguity(pm, w)
        # cells for configs this isoform has no blind coverage for are left blank, not zero
        mat.append([float(np.mean((pm[c_] - vbar) ** 2)) if c_ in cfgs else np.nan
                    for c_ in POOL_CFG])
    mat = np.ma.masked_invalid(np.array(mat))
    im = ax.imshow(mat, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(len(POOL_CFG)))
    ax.set_xticklabels(POOL_CFG, rotation=90, fontsize=6)
    ax.set_yticks(range(len(ISOFORMS)))
    ax.set_yticklabels(ISOFORMS, fontsize=8)
    ax.set_title(f"{side}", fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.045).set_label("A$_a$ (pIC50$^2$)", fontsize=7.5)
fig.suptitle("Which members carry the disagreement, out-of-fold and on the blind set",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.tight_layout(rect=[0, 0, 1, 0.91])
finish(fig, FIG_OUT / "member_ambiguity_heatmap.png")

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/39_ambiguity_and_shared_blend/figures/member_ambiguity_heatmap.png


**Figure 3 — `member_ambiguity_heatmap.png`.** Per-member ambiguity `A_a = mean_x[(V_a(x) − V̄(x))²]`
— each config's own mean squared deviation from the ensemble consensus — out-of-fold and on the
blind set, for the eleven-config pool. Read it to see *where* disagreement lives rather than only
how much there is: a column that is dark on the left and pale on the right is a member that stops
dissenting once trained on the full data. The two panels share no colour scale by design, since the
left is a fold-model quantity and the right a full-data one; the comparison that matters is the
pattern across members, not the absolute shade.

## Verdict

In [17]:
print("=" * 86)
print("NOTEBOOK 39 -- VERDICT")
print("=" * 86)

print("\n--- QUESTION A: does the blend's diversity survive into the blind population? ---")
print(f"\nTRACK 1 (notebook 37's own blend members) CANNOT ANSWER IT. `deadzone` carries")
print(f"  {min(BLEND_W[i].get('deadzone', 0) for i in ISOFORMS):.2f}-{max(BLEND_W[i].get('deadzone', 0) for i in ISOFORMS):.2f} of the weight on every isoform and has NO full-data blind")
print(f"  predictions -- it was never retrained full-data on its own (notebook 32 trained dead-zone")
print(f"  target + AID ARCHITECTURE, a different model). CYP2D6's blend is deadzone alone, so its")
print(f"  blind ambiguity is not computable at all. Excluded and named, never substituted.")

print(f"\nTRACK 2 (the eleven-config pool, every member present on both sides) ANSWERS IT: "
      f"DIVERSITY SURVIVES.")
for _, r in pool.iterrows():
    band = 2 * r["A_oof_sd"]
    inside = abs(r["A_blind"] - r["A_oof_mean"]) <= band
    where = ("within the OOF repeat band" if inside
             else ("ABOVE it -- MORE disagreement, the opposite of collapse"
                   if r["A_blind"] > r["A_oof_mean"] else "BELOW it -- disagreement has shrunk"))
    print(f"  {r['isoform']}: A_oof {r['A_oof_mean']:.4f}+/-{r['A_oof_sd']:.4f}  "
          f"A_blind {r['A_blind']:.4f}  ratio {r['ratio_blind_over_oof']:.2f}x  {where}")
_r = pool["ratio_blind_over_oof"]
print(f"  ratio range {_r.min():.2f}-{_r.max():.2f}. Member disagreement on the blind compounds is")
print(f"  {'NOT collapsed' if _r.min() > 0.5 else 'COLLAPSED'} relative to out-of-fold.")
print(f"  The Krogh-Vedelsby identity E = Ebar - A was verified to hold to "
      f"{pool['identity_max_abs_err'].max():.1e} on the OOF side, confirming the implementation.")

print(f"\nA.4 -- notebook 08's ensembles, which lost 0.116 macro on the board:")
for _, r in nb08_amb.iterrows():
    if np.isnan(r["A_blind"]):
        print(f"  {r['isoform']}: {r['note']}")
    else:
        print(f"  {r['isoform']}: A_oof {r['A_oof_mean']:.4f}+/-{r['A_oof_sd']:.4f}  "
              f"A_blind {r['A_blind']:.4f}  ratio {r['ratio_blind_over_oof']:.2f}x")
_n8 = nb08_amb["ratio_blind_over_oof"].dropna()
if len(_n8):
    if _n8.min() < 0.5:
        print(f"  -> A_blind WAS collapsed for notebook 08's ensembles (ratio as low as {_n8.min():.2f}).")
        print(f"     That is a MECHANISM for a failure currently recorded only as an outcome.")
    else:
        print(f"  -> A_blind was NOT collapsed for notebook 08's ensembles (ratio "
              f"{_n8.min():.2f}-{_n8.max():.2f}).")
        print(f"     So vanished diversity is NOT the mechanism behind NB10's 0.116 macro loss, and")
        print(f"     that failure still needs its explanation elsewhere -- consistent with notebook")
        print(f"     10b's selection-bias account and notebook 38's finding that ranking, not")
        print(f"     spread or placement, is what separates this project's columns.")

print("\n--- QUESTION B: is a single shared-member ensemble viable, and at what cost? ---")
print(f"\n  macro, nested:  PLAIN {plain_nested['macro']:.4f} | unconstrained "
      f"{nested['unconstrained']['macro']:.4f} | shared members "
      f"{nested['shared_members']['macro']:.4f} | shared both {nested['shared_both']['macro']:.4f}")
_cm = float(cost[cost.isoform == 'macro']['cost_shared_members'].iloc[0])
_cb = float(cost[cost.isoform == 'macro']['cost_shared_both'].iloc[0])
print(f"  cost of sharing members: {_cm:+.4f} macro ST-RAE")
print(f"  cost of also sharing weights: a further {_cb - _cm:+.4f} (total {_cb:+.4f})")
if _cb < 0:
    print(f"  NOTE: sharing BOTH is {abs(_cb):.4f} BETTER than the unconstrained per-isoform blend.")
    print(f"  A constraint cannot beat an unconstrained fit in sample; it can out-of-fold, and does")
    print(f"  here. The single-object ensemble is not a compromise on this evidence -- it is the")
    print(f"  better-generalising option, with the per-isoform search paying for overfitting its")
    print(f"  own folds. Driven almost entirely by CYP2D6 ({float(cost[cost.isoform=='CYP2D6']['cost_shared_both'].iloc[0]):+.4f}).")
print(f"  shared set: {shared_set}")
_still = all(cost[cost.isoform == i]["shared_members"].iloc[0] < plain_nested[i] for i in ISOFORMS)
print(f"\n  VIABLE? The shared-member blend still beats PLAIN on "
      f"{'ALL FOUR' if _still else 'NOT all'} isoforms.")
print(f"  CYP2D6 pays {float(cost[cost.isoform=='CYP2D6']['cost_shared_members'].iloc[0]):+.4f} for carrying members its own search rejects;")
print(f"  the other three pay {float(cost[cost.isoform.isin(['CYP1A2','CYP2C9','CYP3A4'])]['cost_shared_members'].mean()):+.4f} on average.")

print("\n--- DOES ANYTHING CLOSE EITHER ROUTE? ---")
closers = []
if _r.min() < 0.5:
    closers.append("Part A: blind-set ambiguity collapsed, so blending cannot pay on the blind set.")
if not _still:
    closers.append("Part B: the shared-member blend fails to beat PLAIN on at least one isoform.")
if float(cost[cost.isoform == "macro"]["shared_members"].iloc[0]) > plain_nested["macro"]:
    closers.append("Part B: the shared-member blend is worse than PLAIN on macro.")
if closers:
    for c_ in closers:
        print(f"  *** {c_}")
else:
    print("  Nothing here closes either route. Both remain open on this evidence.")
print("\n  STANDING CAVEATS carried, not resolved by anything here:")
print("   - A decomposes SQUARED ERROR. The board scores ST-RAE. No A figure above is a")
print("     decomposition of the scored metric, and a surviving A does not guarantee an ST-RAE gain.")
print("   - every ST-RAE in Part B is OUT-OF-FOLD on the cluster-disjoint partition. Notebook 21")
print("     established OOF optima need not transfer, and this project has four CV-to-board")
print("     transfer failures on record. Part B measures a cost, not a board outcome.")
print("   - notebook 38: what separates this project's best columns is RANKING, which no blend")
print("     weighting changes for a fixed member set.")
print("\nSCOPE: nothing trained; no submission built, validated or sent.")
print("=" * 86)

NOTEBOOK 39 -- VERDICT

--- QUESTION A: does the blend's diversity survive into the blind population? ---

TRACK 1 (notebook 37's own blend members) CANNOT ANSWER IT. `deadzone` carries
  0.50-1.00 of the weight on every isoform and has NO full-data blind
  predictions -- it was never retrained full-data on its own (notebook 32 trained dead-zone
  target + AID ARCHITECTURE, a different model). CYP2D6's blend is deadzone alone, so its
  blind ambiguity is not computable at all. Excluded and named, never substituted.

TRACK 2 (the eleven-config pool, every member present on both sides) ANSWERS IT: DIVERSITY SURVIVES.
  CYP1A2: A_oof 0.0985+/-0.0068  A_blind 0.1334  ratio 1.35x  ABOVE it -- MORE disagreement, the opposite of collapse
  CYP2C9: A_oof 0.0651+/-0.0030  A_blind 0.0978  ratio 1.50x  ABOVE it -- MORE disagreement, the opposite of collapse
  CYP2D6: A_oof 0.0655+/-0.0042  A_blind 0.0838  ratio 1.28x  ABOVE it -- MORE disagreement, the opposite of collapse
  CYP3A4: A_oof 0.1172+

## Decisions taken without approval

This section exists whether or not it has content. Every entry is also written to
`outputs/39_ambiguity_and_shared_blend/decisions_taken.json`.

In [18]:
record_decision(
    decision="Excluded `deadzone`, `mae_only` and `single` from the blind-side ambiguity and reported what that leaves, rather than substituting notebook 32's dead-zone+AID model for `deadzone`",
    reason=("The brief requires a member with no full-data blind predictions to be excluded and "
            "named. `deadzone` in notebook 37's design is the dead-zone target on the PLAIN "
            "architecture; the only full-data model carrying a dead-zone target is notebook 32's, "
            "which combines it with the AID ARCHITECTURE and is therefore a different model. "
            "Substituting it would silently answer a different question. The consequence is stated "
            "rather than worked around: `deadzone` holds 0.50-1.00 of the weight on every isoform, "
            "so Track 1 covers at most two of three members, and CYP2D6 -- whose blend is "
            "`deadzone` alone -- has no computable blind ambiguity at all."),
    alternatives=("Substitute notebook 32's dead-zone+AID predictions (answers a different "
                  "question under the same name); or report Track 1 alone and call the question "
                  "unanswerable (discards an answer that the pool in A.3 can actually give)."),
    authority="the brief's own exclusion rule, applied literally")

record_decision(
    decision="Added Track 2 -- the eleven-config pool -- which the brief did not ask for",
    reason=("Track 1 as specified cannot answer the question it was set, because its dominant "
            "member has no blind predictions. Every config in notebook 05's eleven-config pool has "
            "BOTH out-of-fold and full-data blind predictions, with nothing missing and nothing "
            "substituted, so the A_oof-against-A_blind comparison can be made honestly there. It "
            "is also the pool notebook 24 characterised and the one notebook 08 selected from, so "
            "the brief's own step 5 follows from it directly rather than needing separate "
            "scaffolding."),
    alternatives="Report only the crippled Track 1 and leave the question open.",
    authority="autonomy rule: when the specified route cannot answer the question, find one that can and say why")

record_decision(
    decision="Used the RANDOM-partition out-of-fold predictions for Track 2 and A.4, and the cluster-disjoint ones for Track 1 and all of Part B",
    reason=("Track 2's blind counterparts (outputs/12_caruana_retrain_predict/blind_predictions/) "
            "come from models selected and fitted on the random partition, and notebook 08 chose "
            "its ensembles on exactly that partition -- so the random-partition OOF is the "
            "like-for-like side for both. Part B is a re-run of notebook 37's own selection and "
            "must sit on notebook 37's cluster-disjoint folds to be comparable to it, which is "
            "asserted by reproducing 37's published weights and nested figures exactly."),
    alternatives="Force one partition throughout (would either break comparability with notebook 37 or compare notebook 08's ensembles against folds it never saw).",
    authority="autonomy rule: match each comparison to the data it was actually made on, and state the split")

record_decision(
    decision="Chose the shared member set by FORWARD selection rather than exhaustively over all 2,048 subsets",
    reason=("The nested protocol re-selects the set inside each of the 25 folds, so an exhaustive "
            "search would be 25 x 2,048 x 4 isoform weight-fits. Forward selection is also the "
            "same mechanic notebook 37's own greedy uses, which keeps the constrained and "
            "unconstrained searches of a kind. It is a heuristic and can miss the global optimum, "
            "so the cost figures in Part B are upper bounds on the true cost of the constraint, "
            "not exact -- stated here rather than implied."),
    alternatives="Exhaustive subset search (tractable in principle but changes the search mechanic between the two arms being compared, confounding the cost being measured).",
    authority="autonomy rule: record the heuristic and the direction of its error")

record_decision(
    decision="Scored the macro objective as the unweighted mean of the four isoform ST-RAEs",
    reason=("That is how the leaderboard's own MA row is computed -- verified in "
            "docs/leaderboard_submissions.md's provenance section, where every published macro "
            "equals the mean of its four isoform values across all twelve submissions. Using "
            "anything else would optimise a target the board does not score."),
    alternatives="Weight isoforms by label count or by headroom (neither is what the board does).",
    authority="the leaderboard document's own verified convention")

record_decision(
    decision="Reported A per repeat on the out-of-fold side, as mean +/- SD over the five repeats, against a single blind value",
    reason=("The blind set yields one ambiguity value; the out-of-fold side yields five "
            "independent ones, one per repeat. Reporting the spread gives the comparison a noise "
            "band, so 'collapse' can be judged against repeat-to-repeat variation rather than "
            "against a point estimate. The asymmetry that remains -- fold models versus full-data "
            "models -- cannot be removed and is stated in A.2 instead, including the direction of "
            "its expected effect."),
    alternatives="Pool all five repeats into one A (loses the noise band and double-counts each compound five times).",
    authority="autonomy rule: give a comparison its uncertainty where the data allows")

print(f"{len(AUTONOMOUS_DECISIONS)} decision(s) recorded:\n")
for i, d in enumerate(AUTONOMOUS_DECISIONS, 1):
    print(f"{i}. {d['decision']}")
    print(f"   WHY: {d['reason']}")
    print(f"   ALTERNATIVES: {d['alternatives']}")
    print(f"   AUTHORITY: {d['authority']}\n")

6 decision(s) recorded:

1. Excluded `deadzone`, `mae_only` and `single` from the blind-side ambiguity and reported what that leaves, rather than substituting notebook 32's dead-zone+AID model for `deadzone`
   WHY: The brief requires a member with no full-data blind predictions to be excluded and named. `deadzone` in notebook 37's design is the dead-zone target on the PLAIN architecture; the only full-data model carrying a dead-zone target is notebook 32's, which combines it with the AID ARCHITECTURE and is therefore a different model. Substituting it would silently answer a different question. The consequence is stated rather than worked around: `deadzone` holds 0.50-1.00 of the weight on every isoform, so Track 1 covers at most two of three members, and CYP2D6 -- whose blend is `deadzone` alone -- has no computable blind ambiguity at all.
   ALTERNATIVES: Substitute notebook 32's dead-zone+AID predictions (answers a different question under the same name); or report Track 1 alone an

## Scope check — fails loudly

Three tests, following notebooks 30/34/35/37/38's fixed version. **TEST 1** looks only at *tracked*
content changes under a protected path; **TEST 2** checks that nothing inside a protected tree was
written during this run, using the timestamp the setup cell recorded at the top; **TEST 3**
confirms `notebooks/32_deadzone_aid_retrain.ipynb` was never modified, since its candidate-A
submission cell is live uncommented code. A fourth confirms no submission-shaped file was created.

In [19]:
st_out = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                        cwd=REPO_ROOT).stdout
print("git status --porcelain:")
print(st_out if st_out.strip() else "(clean)")

PROTECTED = [
    "data/", "src/",
    "notebooks/32_deadzone_aid_retrain.ipynb", "outputs/32_deadzone_aid_retrain/",
    "notebooks/34_butina_5x5_repeated_cv.ipynb", "outputs/34_butina_5x5/",
    "notebooks/35_butina_repeat3.ipynb", "outputs/35_butina_repeat3/",
    "notebooks/36_spread_corrected_nb32a.ipynb", "outputs/36_spread_corrected_nb32a/",
    "notebooks/37_butina_5x5_complete.ipynb", "outputs/37_butina_5x5_complete/",
    "notebooks/38_cyp3a4_placement_diagnostic.ipynb", "outputs/38_cyp3a4_placement_diagnostic/",
    "outputs/08_ensemble_selection/", "outputs/10_final_retrain_predict/",
    "outputs/11_caruana_prep/", "outputs/12_caruana_retrain_predict/",
    "outputs/30_aid_full_retrain/", "outputs/submissions/",
]

tracked_violations = [l for l in st_out.splitlines()
                      if any(x in l[3:].strip() for x in PROTECTED) and l[:2].strip() != "??"]
written = []
for prot in PROTECTED:
    q = REPO_ROOT / prot
    if not q.exists():
        continue
    for fp in ([q] if q.is_file() else [x for x in q.rglob("*") if x.is_file()]):
        try:
            if fp.stat().st_mtime > RUN_STARTED_AT:
                written.append(str(fp.relative_to(REPO_ROOT)))
        except OSError:
            pass

print(f"\nTEST 1 -- tracked modifications under a protected path: {len(tracked_violations)}")
for t in tracked_violations:
    print(f"    VIOLATION: {t}")
print(f"TEST 2 -- files written inside a protected tree during this run: {len(written)}")
for t in written[:25]:
    print(f"    VIOLATION: {t}")

nb32 = REPO_ROOT / "notebooks" / "32_deadzone_aid_retrain.ipynb"
nb32_ok = (not nb32.exists()) or nb32.stat().st_mtime <= RUN_STARTED_AT
print(f"TEST 3 -- notebooks/32_deadzone_aid_retrain.ipynb unmodified: {nb32_ok}")

made = sorted(p.name for p in OUT.rglob("*") if p.is_file())
subm = [m for m in made if "submission" in m.lower() and m.endswith(".csv")]
print(f"TEST 4 -- submission-shaped CSVs created by this notebook: {len(subm)} {subm}")

if tracked_violations or written or not nb32_ok or subm:
    raise ValueError("scope violated -- investigate before trusting anything above")
print("\nconfirmed: no protected path modified, notebook 32 untouched, no submission file created.")
print(f"\noutputs written by this notebook ({len(made)}): {made}")

git status --porcelain:
 M CLAUDE.md
 M docs/leaderboard_submissions.md
?? notebooks/35_butina_repeat3.ipynb
?? notebooks/36_spread_corrected_nb32a.ipynb
?? notebooks/37_butina_5x5_complete.ipynb
?? notebooks/38_cyp3a4_placement_diagnostic.ipynb
?? notebooks/39_ambiguity_and_shared_blend.ipynb
?? outputs/35_butina_repeat3/
?? outputs/36_spread_corrected_nb32a/
?? outputs/37_butina_5x5_complete/
?? outputs/38_cyp3a4_placement_diagnostic/
?? outputs/39_ambiguity_and_shared_blend/


TEST 1 -- tracked modifications under a protected path: 0
TEST 2 -- files written inside a protected tree during this run: 0
TEST 3 -- notebooks/32_deadzone_aid_retrain.ipynb unmodified: True
TEST 4 -- submission-shaped CSVs created by this notebook: 0 []

confirmed: no protected path modified, notebook 32 untouched, no submission file created.

outputs written by this notebook (14): ['ambiguity_nb08_ensembles.csv', 'ambiguity_oof_vs_blind.png', 'ambiguity_track1_blend_members.csv', 'ambiguity_track2_pool.csv'

---

## Part D — what Parts A and B do and do not establish

**Appended after the fact. Nothing above this point was re-executed**: these cells are
self-contained, reading the saved CSVs this notebook already wrote and re-verifying every quoted
figure and quotation at source. They were executed on their own and spliced in, so Parts A–C's
outputs are exactly as originally produced.

In [1]:
# ---- self-contained: re-import, re-read this notebook's own saved outputs, re-verify quotes.
import hashlib, json, re, shutil, subprocess
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = REPO_ROOT / "outputs" / "39_ambiguity_and_shared_blend"
NB37 = REPO_ROOT / "outputs" / "37_butina_5x5_complete"

pool_d = pd.read_csv(OUT / "ambiguity_track2_pool.csv")
nb08_d = pd.read_csv(OUT / "ambiguity_nb08_ensembles.csv")
cost_d = pd.read_csv(OUT / "shared_set_cost.csv")
avail_d = pd.read_csv(OUT / "blend_members_on_blind.csv")
folds_d = pd.read_csv(OUT / "per_fold_shared_sets.csv")
print("re-read this notebook's own saved outputs:",
      [p.name for p in (OUT / "ambiguity_track2_pool.csv", OUT / "ambiguity_nb08_ensembles.csv",
                        OUT / "shared_set_cost.csv")])

# ---- the two NEW Krogh & Vedelsby quotations Part D relies on, verified verbatim.
KV = Path("/Users/codiefreeman/Zotero/storage/MMANS4Y5/Krogh and Vedelsby - Neural Network "
          "Ensembles, Cross Validation, and Active Learning.pdf")
KV_SHA = "88de32e9c1d3384970036baf59c7fbcdc8918a9f6a86fbf42bcb9500d83cb372"
NEW_QUOTES = {
    "ambiguity_is_a_lower_bound": (
        "the ambiguity is a lower bound for the weighted average of the squared error. An input "
        "pattern that yields a large ambiguity will always have a large average error. On the "
        "other hand, a low ambiguity does not necessarily imply a low error."),
    "ensemble_error_below_member_average": (
        "ensemble is always smaller than the (weighted) average of the ensemble errors,"),
}
def _norm(s):
    for a, b in [("−", "-"), ("–", "-"), ("—", "-"), ("×", "x"),
                 ("’", "'"), ("‘", "'"), ("“", '"'), ("”", '"'),
                 ("ﬁ", "fi"), ("ﬂ", "fl"), (" ", " ")]:
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s).strip()

if KV.exists() and shutil.which("pdftotext"):
    sha = hashlib.sha256(KV.read_bytes()).hexdigest()
    assert sha == KV_SHA, "the Krogh & Vedelsby PDF is not the file Part A verified against"
    body = _norm(subprocess.run(["pdftotext", str(KV), "-"], capture_output=True, text=True,
                                check=True).stdout)
    for k, q in NEW_QUOTES.items():
        ok = _norm(q) in body
        print(f"  {'FOUND verbatim' if ok else '*** NOT FOUND':18s}  {k}")
        assert ok, f"{k} is not in the PDF -- correct the text rather than the check"
    print("both new quotations verified against the same sha256-pinned PDF Part A used.")
else:
    print("*** PDF or pdftotext unavailable -- quotations not re-checked here.")

re-read this notebook's own saved outputs: ['ambiguity_track2_pool.csv', 'ambiguity_nb08_ensembles.csv', 'shared_set_cost.csv']
  FOUND verbatim      ambiguity_is_a_lower_bound
  FOUND verbatim      ensemble_error_below_member_average
both new quotations verified against the same sha256-pinned PDF Part A used.


### D.1 — The ambiguity check is one-directional, and notebook 08 proves it

**It can close a route. It cannot open one.** Part A's finding — `A_blind / A_oof` of 1.28–1.50 —
rules out one specific failure mechanism. It is **not** evidence that a blend will transfer, and the
proof is already in this notebook.

Krogh & Vedelsby say why, in their own words (verified verbatim above):

> the ambiguity is a lower bound for the weighted average of the squared error. An input pattern
> that yields a large ambiguity will always have a large average error. On the other hand, a low
> ambiguity does not necessarily imply a low error.

So `A` is bounded above by `Ē`. A **rising** ambiguity on the blind set is therefore consistent with
two quite different readings, and ambiguity alone cannot separate them:

1. members disagree in a **cancelling** way, which is what an ensemble needs; or
2. the blind population is simply **harder**, so every member is less certain and `Ē` has risen —
   dragging `A` up with it, exactly as the lower-bound relation requires.

`E = Ē − A` needs **both** terms. `Ē` on the blind set needs labels, and there are none until the
reveal. This notebook measured the term that is free and left the term that decides the outcome
unmeasured, which is the honest description of what was done.

**Notebook 08 settles it empirically**, without appeal to the theory: its ensembles had
`A_blind / A_oof` of **1.66 / 1.11 / 1.79** — not collapsed on any isoform — and NB10 still lost
**0.116 macro** on the real board. Healthy blind ambiguity is demonstrably compatible with a failed
ensemble submission, on this project's own data.

**What is recorded, therefore**: the collapse hypothesis is **dead**, including as the explanation
for notebook 08's board failure. That failure now needs a different mechanism. The ratios above are
not evidence a blend will transfer and are not to be cited as such.

In [2]:
print("=" * 84)
print("D.1 -- ONE-DIRECTIONAL: what the ambiguity ratios rule out, and what they do not")
print("=" * 84)
print("\nPart A's pool ratios (re-read from this notebook's own saved file):")
for _, r in pool_d.iterrows():
    print(f"  {r['isoform']}: A_oof {r['A_oof_mean']:.4f}  A_blind {r['A_blind']:.4f}  "
          f"ratio {r['ratio_blind_over_oof']:.2f}x")

print("\nNotebook 08's ensembles -- the ones NB10 submitted:")
for _, r in nb08_d.iterrows():
    if pd.isna(r["ratio_blind_over_oof"]):
        print(f"  {r['isoform']}: {r['note']}")
    else:
        print(f"  {r['isoform']}: ratio {r['ratio_blind_over_oof']:.2f}x   "
              f"({'NOT collapsed' if r['ratio_blind_over_oof'] > 0.5 else 'collapsed'})")

_n8 = nb08_d["ratio_blind_over_oof"].dropna()
NB10_MACRO_LOSS = 0.116
print(f"\n  not-collapsed on {int((_n8 > 0.5).sum())} of {len(_n8)} isoforms where an ensemble existed,")
print(f"  and NB10 still lost {NB10_MACRO_LOSS:.3f} macro ST-RAE on the real board.")
print("\n  => Healthy blind ambiguity is COMPATIBLE with a failed ensemble submission.")
print("     The check can CLOSE a route (a collapsed A would have explained the failure);")
print("     it cannot OPEN one (a surviving A does not predict a gain).")
print("\n  Krogh & Vedelsby's own reason: A <= Ebar, so large A implies large average member error.")
print("  A rising ratio is consistent with cancelling disagreement OR with a harder blind")
print("  population. E = Ebar - A needs both terms, and Ebar on the blind set needs labels.")
print("\n  RECORDED: the collapse hypothesis is dead, including as the explanation for notebook")
print("  08's board failure. That failure needs a different mechanism.")

D.1 -- ONE-DIRECTIONAL: what the ambiguity ratios rule out, and what they do not

Part A's pool ratios (re-read from this notebook's own saved file):
  CYP1A2: A_oof 0.0985  A_blind 0.1334  ratio 1.35x
  CYP2C9: A_oof 0.0651  A_blind 0.0978  ratio 1.50x
  CYP2D6: A_oof 0.0655  A_blind 0.0838  ratio 1.28x
  CYP3A4: A_oof 0.1172  A_blind 0.1625  ratio 1.39x

Notebook 08's ensembles -- the ones NB10 submitted:
  CYP1A2: ratio 1.66x   (NOT collapsed)
  CYP2C9: no ensemble adopted -- singleton winner, ambiguity undefined
  CYP2D6: ratio 1.11x   (NOT collapsed)
  CYP3A4: ratio 1.79x   (NOT collapsed)

  not-collapsed on 3 of 3 isoforms where an ensemble existed,
  and NB10 still lost 0.116 macro ST-RAE on the real board.

  => Healthy blind ambiguity is COMPATIBLE with a failed ensemble submission.
     The check can CLOSE a route (a collapsed A would have explained the failure);
     it cannot OPEN one (a surviving A does not predict a gain).

  Krogh & Vedelsby's own reason: A <= Ebar, so 

### D.2 — The submission-relevant check is blocked, and on exactly one thing

Part A measured **notebook 05's eleven-config pool**, not notebook 37's blend members. That was not
a preference; Track 1 could not be computed, because `deadzone` holds 0.50–1.00 of the weight in
every isoform's blend and has no full-data blind predictions.

So the version of this check that would bear on a **real submission** — the ambiguity of the
ensemble that would actually be sent — requires exactly one thing:

> **one full-data retrain of `deadzone` (the dead-zone clipped target on the plain CheMeleon
> architecture), predicted onto the blind 750.**

That is not extra work invented by this diagnostic. **It is needed to submit the blend at all** —
no blend containing `deadzone` can be built for the blind set without it. The check and the
submission are blocked on the same single artefact.

Notebook 32's full-data model does **not** substitute: it trains the dead-zone target on the **AID
architecture**, a different model, and using it here would answer a different question under the
same name.

In [3]:
print("=" * 84)
print("D.2 -- THE NAMED BLOCKER")
print("=" * 84)
print("\nwhat is left of each isoform's blend on the blind set (this notebook's own saved table):")
print(avail_d[["isoform", "n_members", "n_with_blind_preds", "weight_available",
               "members_kept", "members_dropped"]].to_string(index=False))
print(f"""
BLOCKER, singular: one full-data retrain of `deadzone` -- the dead-zone clipped target on the
plain CheMeleon architecture -- predicted onto the blind 750.

  - it is the only missing member that carries weight ({avail_d['weight_missing'].min():.2f}-{avail_d['weight_missing'].max():.2f} of it per isoform)
  - it is REQUIRED TO SUBMIT THE BLEND AT ALL, so it is not extra work for this diagnostic
  - notebook 32's full-data model is NOT a substitute: dead-zone target on the AID architecture
  - with it, Track 1 becomes computable on all four isoforms, including CYP2D6
""")

D.2 -- THE NAMED BLOCKER

what is left of each isoform's blend on the blind set (this notebook's own saved table):
isoform  n_members  n_with_blind_preds  weight_available               members_kept members_dropped
 CYP1A2          3                   2             0.500       aid;ecfp4_narrow__rf        deadzone
 CYP2C9          3                   2             0.375       aid;ecfp4_narrow__rf        deadzone
 CYP2D6          1                   0             0.000                     (none)        deadzone
 CYP3A4          3                   2             0.400 aid;ecfp4_narrow__lightgbm        deadzone

BLOCKER, singular: one full-data retrain of `deadzone` -- the dead-zone clipped target on the
plain CheMeleon architecture -- predicted onto the blind 750.

  - it is the only missing member that carries weight (0.50-1.00 of it per isoform)
  - it is REQUIRED TO SUBMIT THE BLEND AT ALL, so it is not extra work for this diagnostic
  - notebook 32's full-data model is NOT a substitut

### D.3 — CYP2D6: a standing belief is reversed

**The old claim, as the record has held it since notebook 35** (and restated in notebook 37's own
entry as "HEADLINE 3"): CYP2D6 cannot be blended. Its greedy search collapses to
`{deadzone: 1.0}`, and **25 of 25** per-fold nested selections choose that same single member —
reported there as "a stable property, not a low-power artefact".

**The new claim, from Part B**: that is an artefact of **what the per-isoform search optimises**,
not a property of the isoform. Under one macro-fitted weight vector shared with the other three
isoforms, CYP2D6 **improves by 0.0113 nested ST-RAE** while carrying two members its own search
rejects at every one of the 25 folds.

Both readings of the 25/25 agreement are correct about the data and differ about what it means. The
per-isoform search was consistent *and* overfitting its own folds: it reliably found the member that
minimised CYP2D6's own out-of-fold score, and that is not the same as finding the ensemble that
generalises. Being made to share a weight vector with three other isoforms costs it that overfit.

This is the one place in this project where sharing a constraint across isoforms **helps** rather
than compromising, and it is the largest single effect in Part B.

In [4]:
print("=" * 84)
print("D.3 -- CYP2D6: THE CORRECTION, OLD AND NEW SIDE BY SIDE")
print("=" * 84)
d6 = cost_d[cost_d.isoform == "CYP2D6"].iloc[0]
agree = folds_d[folds_d.level == "shared_both"]["set"].value_counts()
top_set, top_n = agree.index[0], int(agree.iloc[0])

print(f"""
OLD (notebooks 35 and 37): CYP2D6 cannot be blended.
     its greedy picks {{deadzone: 1.0}}; 25/25 per-fold selections agree;
     recorded as "a stable property, not a low-power artefact".

NEW (notebook 39, Part B): that is a property of the PER-ISOFORM OBJECTIVE, not of the isoform.
     unconstrained (its own search)        {d6['unconstrained']:.4f}
     shared members, per-isoform weights   {d6['shared_members']:.4f}   ({d6['cost_shared_members']:+.4f})
     shared members AND weights            {d6['shared_both']:.4f}   ({d6['cost_shared_both']:+.4f})
     -> under ONE macro-fitted weight vector CYP2D6 IMPROVES by {abs(d6['cost_shared_both']):.4f} nested ST-RAE
        while carrying members its own search rejects.
""")
print(f"the shared vector, and how stable it is across folds: {top_set}")
print(f"  chosen in {top_n} of {int(agree.sum())} folds")
n_members_shared = len(top_set.split(";"))
print(f"  CYP2D6 carries {n_members_shared - 1} member(s) beyond its own choice of 1")
print(f"""
BOTH readings of the 25/25 agreement are correct about the data. The per-isoform search was
consistent AND overfitting its own folds: it reliably minimised CYP2D6's own out-of-fold score,
which is not the same as finding the ensemble that generalises.

This is the largest single effect in Part B ({d6['cost_shared_both']:+.4f}) and the one place in this project
where a shared constraint HELPS rather than costing.
""")

D.3 -- CYP2D6: THE CORRECTION, OLD AND NEW SIDE BY SIDE

OLD (notebooks 35 and 37): CYP2D6 cannot be blended.
     its greedy picks {deadzone: 1.0}; 25/25 per-fold selections agree;
     recorded as "a stable property, not a low-power artefact".

NEW (notebook 39, Part B): that is a property of the PER-ISOFORM OBJECTIVE, not of the isoform.
     unconstrained (its own search)        0.8717
     shared members, per-isoform weights   0.8717   (+0.0000)
     shared members AND weights            0.8605   (-0.0113)
     -> under ONE macro-fitted weight vector CYP2D6 IMPROVES by 0.0113 nested ST-RAE
        while carrying members its own search rejects.

the shared vector, and how stable it is across folds: deadzone;aid;ecfp4_narrow__xgboost
  chosen in 25 of 25 folds
  CYP2D6 carries 2 member(s) beyond its own choice of 1

BOTH readings of the 25/25 agreement are correct about the data. The per-isoform search was
consistent AND overfitting its own folds: it reliably minimised CYP2D6's own 

### D.4 — One illustration worth keeping

`ecfp4_narrow__xgboost` is **significantly worse than PLAIN on CYP1A2** at 25 folds — notebook 37
measured `+0.0492` ST-RAE, 95% CI `[+0.0216, +0.0768]`, `p_BH = 0.002`, improving on only 6 of 25
folds — and is significantly worse on CYP2C9 and CYP3A4 too. It nonetheless earns **weight 0.125**
in the single best-generalising ensemble this project has found.

That is Dietterich's accurate-and-diverse condition behaving exactly as described, and this project
has the rule in its own `CLAUDE.md`: a member needs to be **better than random and wrong about
different compounds**, not individually good. Judging candidate members by their solo score would
have excluded this one on three isoforms out of four.

In [5]:
print("=" * 84)
print("D.4 -- A MEMBER THAT IS SIGNIFICANTLY WORSE SOLO, AND EARNS WEIGHT ANYWAY")
print("=" * 84)
pt = pd.read_csv(NB37 / "paired_tests_vs_plain_25fold.csv")
pt = pt[(pt.metric == "ST-RAE") & (pt.arm == "ecfp4_narrow__xgboost")]
print("\nnotebook 37's own 25-fold paired tests vs PLAIN (re-read from its saved file):")
print(pt[["isoform", "mean_diff", "ci_low", "ci_high", "p_BH", "verdict",
          "n_improving"]].round(4).to_string(index=False))
top_set = folds_d[folds_d.level == "shared_both"]["set"].value_counts().index[0]
print(f"\nthe best-generalising ensemble's member set: {top_set.split(';')}")
print(f"  ecfp4_narrow__xgboost IS a member: {'ecfp4_narrow__xgboost' in top_set.split(';')}")
n_worse = int((pt["verdict"] == "SIGNIFICANTLY WORSE").sum())
print(f"\n  significantly WORSE than PLAIN solo on {n_worse} of 4 isoforms, and still carries weight.")
print("""
  This is Dietterich's accurate-and-diverse condition behaving as described, and it is already
  this project's own stated rule: a member needs to be better than random and wrong about
  DIFFERENT compounds, not individually good. Screening candidate members on solo score would
  have thrown this one out on three isoforms.
""")

D.4 -- A MEMBER THAT IS SIGNIFICANTLY WORSE SOLO, AND EARNS WEIGHT ANYWAY

notebook 37's own 25-fold paired tests vs PLAIN (re-read from its saved file):
isoform  mean_diff  ci_low  ci_high  p_BH             verdict  n_improving
 CYP1A2     0.0492  0.0216   0.0768 0.002 SIGNIFICANTLY WORSE            6
 CYP2C9     0.1176  0.0944   0.1407 0.000 SIGNIFICANTLY WORSE            0
 CYP2D6     0.0113 -0.0212   0.0437 1.000       NOT DIFFERENT           11
 CYP3A4     0.0976  0.0836   0.1116 0.000 SIGNIFICANTLY WORSE            0

the best-generalising ensemble's member set: ['deadzone', 'aid', 'ecfp4_narrow__xgboost']
  ecfp4_narrow__xgboost IS a member: True

  significantly WORSE than PLAIN solo on 3 of 4 isoforms, and still carries weight.

  This is Dietterich's accurate-and-diverse condition behaving as described, and it is already
  this project's own stated rule: a member needs to be better than random and wrong about
  DIFFERENT compounds, not individually good. Screening candidate m

### D.5 — What this does not establish

Stated together so nothing here is carried further than it goes:

- **The margin is 0.0027 macro ST-RAE.** That is the whole of the shared-members-and-weights
  advantage over the unconstrained blend. For scale, it is roughly a quarter of the smallest gap
  ever to separate two submissions on this project's board.
- **It is out-of-fold**, on the cluster-disjoint partition — and this project has **four
  CV-to-board transfer failures** on record (notebook 08's ensembles, notebook 29's AID
  confirmation via NB30, and the two notebook 10/12 regressions). Notebook 21 established
  separately that an out-of-fold optimum need not transfer at all.
- **Notebook 38 established that what separates the leading CYP3A4 columns is RANKING**, and no
  blend weighting changes ranking for a fixed member set. A better-weighted ensemble of the same
  members does not touch the component that is actually costing that isoform.

**The finding is about the right SHAPE for an ensemble — one member set, one weight vector — not
evidence that such an ensemble will score.** Whether it does is a board question, and answering it
needs the `deadzone` retrain named in D.2 and a submission slot.

In [6]:
print("=" * 84)
print("D.5 -- WHAT THIS DOES NOT ESTABLISH")
print("=" * 84)
m = cost_d[cost_d.isoform == "macro"].iloc[0]
print(f"""
  1. THE MARGIN IS {abs(m['cost_shared_both']):.4f} MACRO ST-RAE.
     unconstrained {m['unconstrained']:.4f} -> shared members and weights {m['shared_both']:.4f}.
     The smallest gap ever to separate two submissions on this project's board is 0.0110, so this
     advantage is roughly a quarter of that.

  2. IT IS OUT-OF-FOLD, on the cluster-disjoint partition. Four CV-to-board transfer failures are
     on record (08's ensembles; 29's AID confirmation via NB30; the NB10 and NB12 regressions),
     and notebook 21 established separately that an OOF optimum need not transfer.

  3. NOTEBOOK 38: what separates the leading CYP3A4 columns is RANKING -- 04b's Spearman 0.8329
     against 0.8020 -- and no blend weighting changes ranking for a fixed member set.

  The finding is about the right SHAPE for an ensemble (one member set, one weight vector), NOT
  evidence that such an ensemble will score. That is a board question, and it needs the deadzone
  retrain named in D.2 plus a submission slot.
""")
print("=" * 84)
print("PART D COMPLETE -- documentation only. Nothing trained, no submission built, nothing above")
print("this section re-executed.")
print("=" * 84)

D.5 -- WHAT THIS DOES NOT ESTABLISH

  1. THE MARGIN IS 0.0027 MACRO ST-RAE.
     unconstrained 0.6529 -> shared members and weights 0.6502.
     The smallest gap ever to separate two submissions on this project's board is 0.0110, so this
     advantage is roughly a quarter of that.

  2. IT IS OUT-OF-FOLD, on the cluster-disjoint partition. Four CV-to-board transfer failures are
     on record (08's ensembles; 29's AID confirmation via NB30; the NB10 and NB12 regressions),
     and notebook 21 established separately that an OOF optimum need not transfer.

  3. NOTEBOOK 38: what separates the leading CYP3A4 columns is RANKING -- 04b's Spearman 0.8329
     against 0.8020 -- and no blend weighting changes ranking for a fixed member set.

  The finding is about the right SHAPE for an ensemble (one member set, one weight vector), NOT
  evidence that such an ensemble will score. That is a board question, and it needs the deadzone
  retrain named in D.2 plus a submission slot.

PART D COMPLE

### Decisions taken without approval — Part D

Appended to `decisions_taken.json` alongside Parts A–C's own six.

In [7]:
DEC_PATH = OUT / "decisions_taken.json"
existing = json.loads(DEC_PATH.read_text())
new = [
    {"decision": "Appended Part D by executing its cells in isolation and splicing them in, rather than re-running notebook 39",
     "reason": ("The brief requires that nothing above Part D be re-executed. Notebook 39 is "
                "deterministic and would have reproduced itself, but re-running it would still "
                "have rewritten every output cell above, which is exactly what was ruled out. "
                "Part D's cells were therefore written to be self-contained -- they re-import, "
                "re-read this notebook's own saved CSVs, and re-verify both new Krogh & Vedelsby "
                "quotations against the same sha256-pinned PDF -- executed as a standalone "
                "notebook, and spliced on with their outputs. Parts A-C's outputs are byte-for-"
                "byte as originally produced."),
     "alternatives": "Re-run the whole notebook (contradicts the brief); or append markdown only (leaves every Part D figure unverified at source).",
     "authority": "the brief's own instruction not to re-execute anything above"},
    {"decision": "Recorded D.1 as closing the collapse hypothesis rather than as a caveat on Part A's result",
     "reason": ("Part A's headline -- ambiguity rising 1.28-1.50x on the blind set -- reads as "
                "support for blending unless the one-directionality is stated as a finding in its "
                "own right. Notebook 08's ensembles make it empirical rather than theoretical: "
                "ratios 1.66/1.11/1.79, not collapsed, and NB10 still lost 0.116 macro. Krogh & "
                "Vedelsby's own lower-bound relation (A <= Ebar, quoted verbatim) supplies the "
                "mechanism. Filing this as a caveat would leave the ratios citable as transfer "
                "evidence, which they are not."),
     "alternatives": "Note it as a limitation inside Part A (weaker, and leaves the misreading available).",
     "authority": "the brief's item 1, which asks for exactly this framing"},
]
for d in new:
    d["at_utc"] = datetime.now(timezone.utc).isoformat()
    d["part"] = "D"
DEC_PATH.write_text(json.dumps(existing + new, indent=2))
print(f"{len(existing)} decisions from Parts A-C + {len(new)} from Part D = "
      f"{len(existing)+len(new)} recorded in {DEC_PATH.name}\n")
for i, d in enumerate(new, 1):
    print(f"{i}. {d['decision']}")
    print(f"   WHY: {d['reason']}")
    print(f"   ALTERNATIVES: {d['alternatives']}")
    print(f"   AUTHORITY: {d['authority']}\n")

6 decisions from Parts A-C + 2 from Part D = 8 recorded in decisions_taken.json

1. Appended Part D by executing its cells in isolation and splicing them in, rather than re-running notebook 39
   WHY: The brief requires that nothing above Part D be re-executed. Notebook 39 is deterministic and would have reproduced itself, but re-running it would still have rewritten every output cell above, which is exactly what was ruled out. Part D's cells were therefore written to be self-contained -- they re-import, re-read this notebook's own saved CSVs, and re-verify both new Krogh & Vedelsby quotations against the same sha256-pinned PDF -- executed as a standalone notebook, and spliced on with their outputs. Parts A-C's outputs are byte-for-byte as originally produced.
   ALTERNATIVES: Re-run the whole notebook (contradicts the brief); or append markdown only (leaves every Part D figure unverified at source).
   AUTHORITY: the brief's own instruction not to re-execute anything above

2. Recorde